# PA5/PA6 HWP analysis pipeline

This notebook is a reusable end-to-end pipeline for complementary co-pol and x-pol TeraFlash HWP angle scans. For each new measurement run:

1. Put the co-pol, x-pol, and open-reference measurements in one measurement folder.
2. Edit only **Section 2: User configuration**.
3. Run **Restart Kernel and Run All**.
4. Read the printed run summary and the CSV/PNG/JSON exports in `results/<RUN_NAME>/`.

The pipeline calculates:

- gain-matched normalized complementary-channel Stokes-q-like angle sweeps;
- fourth-harmonic fits, `d(theta) = c0 + c4*cos(4*theta) + s4*sin(4*theta)`;
- modulation efficiency from the fitted fourth-harmonic amplitude;
- formal covariance-derived fit uncertainties;
- residual and fit-quality diagnostics;
- PA5/F090 and PA6/F150 band summaries;
- normalized Stokes-q HWP-angle phase;
- optional raw co-pol/x-pol Fourier diagnostics;
- an idealized Kevin Figure 4.8 Mueller-model convention comparison for +58 deg and -58 deg central-plate orientations.

Scientific lineage:

- The modulation-efficiency calculation is retained from `pa5_hwp-ModulationEfficiency_0622.ipynb`.
- The normalized Stokes-q phase and Kevin Figure 4.8 convention diagnostic are retained from the phase/convention notebooks.
- Kevin Coughlin's thesis is used for Eq. 4.11 normalized q, Eq. 4.12 relative gain, the fourth-harmonic interpretation, Figure 4.8, and the three-stack Pancharatnam design with an approximately 58 deg central plate.
- Braithwaite is used only as a physical cross-check that complementary co-pol and cross-pol measurements diagnose HWP modulation; its NIKA2-specific equations are not substituted into this PA5/PA6 calculation.

The notebook intentionally centralizes convention-sensitive choices. In particular, the supplied notebooks contain stale text about `+phase/4` versus `-phase/4`. The executed code in the corrected phase notebook uses `+atan2(s4,c4)/4`, wrapped modulo 90 deg. That behavior is preserved here as the default `PHASE_ANGLE_SIGN = +1.0`; changing it to `-1.0` mirrors the stage-coordinate phase trend.


## 1. Imports and plotting setup

This cell uses only standard scientific Python packages plus the project-specific `TeraToptica` analyzer. A missing `TeraToptica` import is reported here and raised clearly at the loading stage, so configuration and data-discovery cells can still be inspected.


In [ ]:
from __future__ import annotations

import json
import os
import platform
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable
import importlib.metadata as importlib_metadata

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

try:
    from IPython.display import display
except Exception:  # pragma: no cover - plain Python fallback
    def display(obj: Any) -> None:
        print(obj)

plt.style.use('default')
mpl.rcParams.update({
    'font.family': 'serif',
    'axes.grid': False,
    'figure.dpi': 100,
    'savefig.dpi': 300,
    'axes.spines.top': True,
    'axes.spines.right': True,
})

try:
    from TeraToptica import TeraFlashAnalyzer, TeraFlashConfig
    TERATOPTICA_AVAILABLE = True
    TERATOPTICA_IMPORT_ERROR = None
except ModuleNotFoundError as exc:
    TeraFlashAnalyzer = None
    TeraFlashConfig = None
    TERATOPTICA_AVAILABLE = False
    TERATOPTICA_IMPORT_ERROR = exc
    print(
        'TeraToptica is not importable in this kernel. Data-independent cells can '
        'be inspected, but the loading cell will raise a clear RuntimeError before '
        'analysis. Original import error: '
        f'{exc}'
    )
except Exception as exc:
    TeraFlashAnalyzer = None
    TeraFlashConfig = None
    TERATOPTICA_AVAILABLE = False
    TERATOPTICA_IMPORT_ERROR = exc
    print(
        'TeraToptica import failed. The loading cell will raise a clear RuntimeError. '
        f'Original error: {type(exc).__name__}: {exc}'
    )

analysis_warnings: list[str] = []
figures_to_save: dict[str, mpl.figure.Figure] = {}
exported_files: list[Path] = []
pipeline_stage_status: dict[str, bool] = {}


## 2. User configuration

Edit this cell for a new measurement run. Routine run-specific settings should live here rather than in later analysis cells.


In [ ]:
# ===================== USER CONFIGURATION: EDIT THIS CELL ONLY =====================

from pathlib import Path

RUN_NAME = 'file name'

DATA_ROOT = Path("data")
DATA_DIR = DATA_ROOT / '20260731_reversed'

RESULTS_DIR = DATA_ROOT / 'results' / RUN_NAME

# Measurement naming convention. With the defaults, angle scans are named like
# copol_90deg_2400-2600ps_100avgs and xpol_90deg_2400-2600ps_100avgs.
CO_POL_FILENAME_STEM = 'copol'
X_POL_FILENAME_STEM = 'xpol'
OPEN_REFERENCE_STEM = 'copol_open'
OPEN_REFERENCE_NAME = None  # If None, use f'{OPEN_REFERENCE_STEM}{SCAN_SUFFIX}'.
SCAN_SUFFIX = '_3160-3360ps_100avgs'

SHOW_ERROR_BARS = False

# HWP angle sampling.
START_ANGLE_DEG = 0.0
STOP_ANGLE_DEG = 360.0
ANGLE_STEP_DEG = 5.0

# Map the recorded stage labels into one fixed physical/lab angle convention.
# +1.0: recorded 0, 5, 10, ... is physically +0, +5, +10, ...
# -1.0: recorded 0, 5, 10, ... is physically  0, -5, -10, ...
ANGLE_DIRECTION_SIGN = 1.0

# Frequency samples used for the modulation-efficiency fit summary.
MODULATION_FREQUENCY_START_GHZ = 50.0
MODULATION_FREQUENCY_STOP_GHZ = 200.0
MODULATION_FREQUENCY_STEP_GHZ = 1.0

# Frequency samples used for the Stokes-q phase fit summary.
PHASE_FREQUENCY_START_GHZ = 50.0
PHASE_FREQUENCY_STOP_GHZ = 250.0
PHASE_FREQUENCY_STEP_GHZ = 1.0

# Representative frequencies used in angle-sweep figures.
REPRESENTATIVE_FREQUENCIES_GHZ = [90.0, 120.0, 150.0]
PHASE_REPRESENTATIVE_FREQUENCIES_GHZ = [75.0, 125.0, 175.0]

# Designed bands.
PA5_F090_BAND_GHZ = (77.0, 112.0)
PA6_F150_BAND_GHZ = (124.0, 172.0)

# Spectrum key and transmission convention.
SIGNAL_KEY = 't'
TRANSMISSION_POWER_CONVENTION = 'signal_is_power_transmission'
# Allowed values:
# - 'signal_is_power_transmission': preserve the current executable notebooks' use_power=False behavior.
# - 'signal_is_field_amplitude': square the selected signal before forming detected power.

# Gain matching and phase convention.
MODULATION_USE_GAIN_MATCHING = True
PHASE_USE_GAIN_MATCHING = True
PHASE_ANGLE_SIGN = 1.0
PHASE_ZERO_OFFSET_DEG = 0.0
PHASE_WRAP_PERIOD_DEG = 90.0

# Optional analysis blocks.
RUN_RAW_CHANNEL_DIAGNOSTIC = True
RUN_KEVIN_MODEL_COMPARISON = True
RUN_MODULATION_MODEL_COMPARISON = True
USE_PHASE_QUALITY_MASK_FOR_MODEL_COMPARISON = True
SAVE_FIGURES = True

# Optional digitized Kevin Figure 4.21 modulation-efficiency model. If absent, the
# measured-vs-ideal plot still runs and a warning is recorded.
MF_MODEL_CSV = Path("data/figure_4_21_modeled_only.csv")

# Optional backward-compatibility preview comparison. Leave empty for portable runs.
# If paths are supplied, this reads saved display previews from source notebooks and
# compares the visible rows against the new run.
SOURCE_NOTEBOOK_PATHS: list[Path] = []

# TeraFlash analyzer settings retained from the validated notebooks.
MASK_BOUNDS = (
    (0, 50),
    (545, 570), (740, 770), (1080, 1245), (1390, 1430),
    (1585, 1810), (1855, 1950),
    (2000, 10000),
)
TERAFLASH_WINDOW = 'boxcar'
TERAFLASH_WINDOW_SIZE = 25.0
TERAFLASH_LAYOUT = 'separate_files'

# Numerical and diagnostic thresholds.
EXPECTED_EFFICIENCY = 1.0
DENOMINATOR_RELATIVE_FLOOR = 1e-12
MIN_VALID_ANGLES_FOR_FIT = 4
SMALL_FOURTH_HARMONIC_AMPLITUDE = 1e-6
MAX_RESIDUAL_RMS_WARNING = 0.08
GAIN_RELATIVE_WARNING = 0.25
COUNTERPHASE_TOLERANCE_DEG = 30.0
DATA_DISCOVERY_RECURSIVE = False
FIGURE_DPI = 300

# Plot colors retained from the source notebooks.
CO_COLOR = '#800000'
X_COLOR = '#000080'
BAND_SHADE_COLOR = 'k'

# Derived aliases used by legacy source-code-derived sections.
USE_POWER = TRANSMISSION_POWER_CONVENTION == 'signal_is_field_amplitude'
SIGNAL_ERROR_KEY = 'err' if SIGNAL_KEY == 't' else None
designed_band_ghz_f090 = PA5_F090_BAND_GHZ
designed_band_ghz_f150 = PA6_F150_BAND_GHZ
co_color = CO_COLOR
x_color = X_COLOR


## 3. Data naming, discovery, and validation

This section validates the measurement folder before TeraFlash loading begins. A measurement base name may correspond to several physical files; the discovery report therefore counts matching filesystem entries rather than assuming one file per scan.


In [ ]:
def warn_pipeline(message: str) -> None:
    """Record an analysis warning and show it through Python's warnings system."""
    if message not in analysis_warnings:
        analysis_warnings.append(message)
    warnings.warn(message, RuntimeWarning, stacklevel=2)


def frequency_grid(start_ghz: float, stop_ghz: float, step_ghz: float) -> np.ndarray:
    """Build an inclusive frequency grid with floating-point endpoint tolerance."""
    if step_ghz <= 0:
        raise ValueError(f'Frequency step must be positive, got {step_ghz}.')
    if stop_ghz < start_ghz:
        raise ValueError(
            f'Frequency stop {stop_ghz} GHz is below start {start_ghz} GHz.'
        )
    count = int(np.floor((stop_ghz - start_ghz) / step_ghz + 1e-9)) + 1
    values = start_ghz + step_ghz * np.arange(count, dtype=float)
    if values.size == 0 or values[-1] < stop_ghz - 1e-9:
        values = np.append(values, stop_ghz)
    return np.round(values, 12)


def angle_grid(start_deg: float, stop_deg: float, step_deg: float) -> np.ndarray:
    """Build an inclusive HWP angle grid from the configured run sampling."""
    if step_deg <= 0:
        raise ValueError(f'Angle step must be positive, got {step_deg}.')
    if stop_deg < start_deg:
        raise ValueError(f'Angle stop {stop_deg} deg is below start {start_deg} deg.')
    count = int(np.floor((stop_deg - start_deg) / step_deg + 1e-9)) + 1
    values = start_deg + step_deg * np.arange(count, dtype=float)
    if values.size == 0 or values[-1] < stop_deg - 1e-9:
        values = np.append(values, stop_deg)
    return np.round(values, 12)


def format_angle_for_filename(angle_deg: float) -> str:
    """Format an angle as used in the TeraFlash measurement names."""
    angle = float(angle_deg)
    if np.isclose(angle, round(angle), atol=1e-9):
        return str(int(round(angle)))
    return f'{angle:g}'


def expected_measurement_base_name(channel_stem: str, angle_deg: float) -> str:
    """Construct the expected measurement base name for one channel and angle."""
    return f'{channel_stem}_{format_angle_for_filename(angle_deg)}deg{SCAN_SUFFIX}'


def expected_open_reference_base_name() -> str:
    """Return the configured open-reference measurement base name."""
    if OPEN_REFERENCE_NAME:
        return str(OPEN_REFERENCE_NAME)
    return f'{OPEN_REFERENCE_STEM}{SCAN_SUFFIX}'


def measurement_file_matches_base(path: Path, base_name: str) -> bool:
    """Return True if a filesystem entry plausibly belongs to a measurement base."""
    name = path.name
    return (
        name == base_name
        or path.stem == base_name
        or name.startswith(base_name + '.')
        or name.startswith(base_name + '_')
    )


def locate_measurement_files(
    data_dir: Path,
    base_name: str,
    recursive: bool = DATA_DISCOVERY_RECURSIVE,
) -> list[Path]:
    """Locate all physical entries associated with a TeraFlash measurement base."""
    data_dir = Path(data_dir).expanduser()
    if not data_dir.exists():
        return []
    iterator = data_dir.rglob('*') if recursive else data_dir.iterdir()
    matches = [path for path in iterator if measurement_file_matches_base(path, base_name)]
    return sorted(matches, key=lambda p: str(p))


def duplicate_measurement_groups(matches: list[Path]) -> list[str]:
    """Identify suspicious duplicate groups while allowing multi-file measurements."""
    if not matches:
        return []
    parents = sorted({str(path.parent) for path in matches})
    duplicate_notes: list[str] = []
    if len(parents) > 1:
        duplicate_notes.append('matches appear in multiple directories: ' + '; '.join(parents))
    directory_matches = [path for path in matches if path.is_dir()]
    if len(directory_matches) > 1:
        duplicate_notes.append(
            'multiple matching measurement directories: '
            + '; '.join(str(path) for path in directory_matches)
        )
    return duplicate_notes


def validate_measurement_set() -> dict[str, Any]:
    """Validate the configured data directory and expected measurement names."""
    data_dir = Path(DATA_DIR).expanduser()
    if not data_dir.exists():
        raise FileNotFoundError(
            f'Data directory not found: {data_dir}\n'
            'Edit DATA_ROOT/DATA_DIR in Section 2 or set HWP_DATA_PATH.'
        )
    if not data_dir.is_dir():
        raise NotADirectoryError(f'DATA_DIR is not a directory: {data_dir}')

    expected_angles_deg = angle_grid(START_ANGLE_DEG, STOP_ANGLE_DEG, ANGLE_STEP_DEG)
    expected_angle_keys = [format_angle_for_filename(angle) for angle in expected_angles_deg]
    open_reference_base = expected_open_reference_base_name()
    open_reference_matches = locate_measurement_files(data_dir, open_reference_base)

    rows: list[dict[str, Any]] = []
    missing: list[str] = []
    duplicate_notes: list[str] = []

    def append_record(channel: str, angle_deg: float | None, base_name: str) -> None:
        matches = locate_measurement_files(data_dir, base_name)
        notes = duplicate_measurement_groups(matches)
        if not matches:
            missing.append(base_name)
        if notes:
            duplicate_notes.extend(f'{base_name}: {note}' for note in notes)
        rows.append({
            'channel': channel,
            'angle_deg': np.nan if angle_deg is None else float(angle_deg),
            'measurement_base_name': base_name,
            'n_matching_entries': len(matches),
            'matching_entries': '; '.join(str(path.relative_to(data_dir)) for path in matches[:8]),
            'duplicate_notes': ' | '.join(notes),
        })

    append_record('open-reference', None, open_reference_base)
    for angle_deg in expected_angles_deg:
        append_record('co-pol', float(angle_deg), expected_measurement_base_name(CO_POL_FILENAME_STEM, angle_deg))
        append_record('x-pol', float(angle_deg), expected_measurement_base_name(X_POL_FILENAME_STEM, angle_deg))

    discovery_df = pd.DataFrame(rows)
    co_count = int((discovery_df['channel'] == 'co-pol').sum())
    x_count = int((discovery_df['channel'] == 'x-pol').sum())
    co_missing = int(((discovery_df['channel'] == 'co-pol') & (discovery_df['n_matching_entries'] == 0)).sum())
    x_missing = int(((discovery_df['channel'] == 'x-pol') & (discovery_df['n_matching_entries'] == 0)).sum())

    print('Data-discovery summary')
    print(f'  Data directory: {data_dir}')
    print(f'  Open reference base: {open_reference_base}')
    print(f'  Open reference matching entries: {len(open_reference_matches)}')
    print(f'  Expected co-pol scans: {co_count}; missing: {co_missing}')
    print(f'  Expected x-pol scans: {x_count}; missing: {x_missing}')
    print(f'  Angle sampling: {expected_angles_deg[0]:g} to {expected_angles_deg[-1]:g} deg in {ANGLE_STEP_DEG:g} deg steps')

    if duplicate_notes:
        warn_pipeline(
            'Potential duplicate measurement groups detected. Review discovery_df. '
            + ' || '.join(duplicate_notes[:8])
        )

    if missing:
        preview = '\n'.join(f'  - {name}' for name in missing[:25])
        more = '' if len(missing) <= 25 else f'\n  ... and {len(missing) - 25} more'
        raise FileNotFoundError(
            'Missing required measurement bases before loading begins:\n'
            f'{preview}{more}'
        )

    return {
        'data_dir': data_dir,
        'expected_angles_deg': expected_angles_deg,
        'expected_angle_keys': expected_angle_keys,
        'open_reference_base_name': open_reference_base,
        'discovery_df': discovery_df,
        'missing_measurements': missing,
        'duplicate_notes': duplicate_notes,
    }


discovery = validate_measurement_set()
discovery_df = discovery['discovery_df']
display(discovery_df.head(12))
pipeline_stage_status['data_discovery'] = True


## 4. TeraFlash configuration and loading

This section preserves the validated `TeraFlashConfig` settings, mask bounds, separate-file layout, smoothing, and `TeraFlashAnalyzer(...).get_smoothed_spectra()` call pattern. Loading failures include the channel, angle, measurement base name, data path, and original exception.


In [ ]:
if not TERATOPTICA_AVAILABLE:
    raise RuntimeError(
        'TeraToptica is required for TeraFlash data loading but is unavailable in this kernel. '
        'Install or activate the project-specific TeraToptica package, then restart and run all. '
        f'Original import error: {type(TERATOPTICA_IMPORT_ERROR).__name__}: {TERATOPTICA_IMPORT_ERROR}'
    )

tf_cfg = TeraFlashConfig(
    window=TERAFLASH_WINDOW,
    window_size=TERAFLASH_WINDOW_SIZE,
    mask_bounds=MASK_BOUNDS,
    layout=TERAFLASH_LAYOUT,
)

degrees = discovery['expected_angles_deg']
degrees_str = discovery['expected_angle_keys']
open_reference_file = discovery['open_reference_base_name']


def teraflash_data_path_string(data_dir: Path) -> str:
    """Return a TeraFlash-compatible directory string with a trailing separator."""
    path_string = str(Path(data_dir).expanduser())
    return path_string if path_string.endswith(os.sep) else path_string + os.sep


def load_one_smoothed_spectrum(
    channel_label: str,
    angle_deg: float,
    measurement_base_name: str,
    data_dir: Path,
    open_reference_base_name: str,
) -> dict[str, Any]:
    """Load one TeraFlash scan and return the source notebook's spectrum fields."""
    try:
        f, t, ut, mask, err, boxnum = TeraFlashAnalyzer(
            teraflash_data_path_string(data_dir),
            measurement_base_name,
            open_reference_base_name,
            config=tf_cfg,
        ).get_smoothed_spectra()
    except Exception as exc:
        raise RuntimeError(
            'Failed to load TeraFlash scan.\n'
            f'  channel: {channel_label}\n'
            f'  HWP angle: {angle_deg:g} deg\n'
            f'  measurement base name: {measurement_base_name}\n'
            f'  data path: {data_dir}\n'
            f'  open reference: {open_reference_base_name}\n'
            f'  original exception: {type(exc).__name__}: {exc}'
        ) from exc
    return {
        'f': f,
        't': t,
        'ut': ut,
        'mask': mask,
        'err': err,
        'boxnum': boxnum,
        'measurement_base_name': measurement_base_name,
    }


def load_angle_scans(channel_stem: str, channel_label: str) -> dict[str, dict[str, Any]]:
    """Load every configured HWP angle scan for one polarization channel."""
    channel_data: dict[str, dict[str, Any]] = {}
    for angle_deg, angle_key in zip(degrees, degrees_str):
        measurement_base_name = expected_measurement_base_name(channel_stem, angle_deg)
        channel_data[str(angle_key)] = load_one_smoothed_spectrum(
            channel_label=channel_label,
            angle_deg=float(angle_deg),
            measurement_base_name=measurement_base_name,
            data_dir=discovery['data_dir'],
            open_reference_base_name=open_reference_file,
        )
    return channel_data


s_data = load_angle_scans(CO_POL_FILENAME_STEM, 'co-pol')
p_data = load_angle_scans(X_POL_FILENAME_STEM, 'x-pol')
print(f'Loaded {len(s_data)} co-pol and {len(p_data)} x-pol angle scans.')
pipeline_stage_status['teraflash_loading'] = True


## 5. Shared interpolation, normalization, and fitting helpers

These helpers consolidate the validated implementations from the source notebooks. The scientific definitions are unchanged: exact requested frequencies are obtained by finite linear interpolation without extrapolation; the normalized complementary-channel projection uses an optional relative gain term; and the same full fourth-harmonic model is used for modulation efficiency and Stokes-q phase.


In [ ]:
def _interpolate_finite(
    frequencies: Iterable[float],
    values: Iterable[float],
    target_frequency_ghz: float,
) -> float:
    """Interpolate one finite spectrum without extrapolating."""
    f = np.asarray(frequencies, dtype=float)
    y = np.asarray(values, dtype=float)
    finite = np.isfinite(f) & np.isfinite(y)
    if np.count_nonzero(finite) < 2:
        return np.nan

    f = f[finite]
    y = y[finite]
    order = np.argsort(f)
    f = f[order]
    y = y[order]

    # np.interp expects increasing x values. Keep the first value at duplicates.
    f, unique_indices = np.unique(f, return_index=True)
    y = y[unique_indices]
    if f.size < 2 or target_frequency_ghz < f[0] or target_frequency_ghz > f[-1]:
        return np.nan
    return float(np.interp(target_frequency_ghz, f, y))


def interpolate_angle_sweep(
    data_dict: dict[str, dict[str, Any]],
    angle_keys: Iterable[Any],
    frequency_ghz: float,
    key: str = 't',
    error_key: str | None = 'err',
) -> tuple[np.ndarray, np.ndarray | None]:
    """Interpolate one channel at an exact frequency across all HWP angles."""
    angle_keys = list(angle_keys)
    values = np.full(len(angle_keys), np.nan, dtype=float)
    errors = None if error_key is None else np.full(len(angle_keys), np.nan, dtype=float)

    for index, angle_key in enumerate(angle_keys):
        scan = data_dict[str(angle_key)]
        values[index] = _interpolate_finite(scan['f'], scan[key], frequency_ghz)
        if errors is not None and error_key in scan:
            errors[index] = _interpolate_finite(scan['f'], scan[error_key], frequency_ghz)
    return values, errors


def to_detected_power(
    signal: Iterable[float],
    sigma_signal: Iterable[float] | None = None,
    use_power: bool = True,
) -> tuple[np.ndarray, np.ndarray | None]:
    """Convert amplitude transmission to power, with first-order uncertainty."""
    signal = np.asarray(signal, dtype=float)
    if use_power:
        power = signal**2
        sigma_power = (
            None
            if sigma_signal is None
            else 2.0 * np.abs(signal) * np.asarray(sigma_signal, dtype=float)
        )
    else:
        power = signal.copy()
        sigma_power = (
            None
            if sigma_signal is None
            else np.asarray(sigma_signal, dtype=float).copy()
        )
    return power, sigma_power


def construct_normalized_linear_stokes(
    co_data: dict[str, dict[str, Any]],
    x_data: dict[str, dict[str, Any]],
    angle_keys: Iterable[Any],
    frequency_ghz: float,
    signal_key: str = 't',
    error_key: str | None = 'err',
    use_power: bool = True,
    use_gain_matching: bool = True,
    denominator_relative_floor: float = 1e-12,
) -> dict[str, Any]:
    """Build the gain-matched normalized linear-Stokes projection."""
    angle_keys = list(angle_keys)
    co_signal, sigma_co_signal = interpolate_angle_sweep(
        co_data, angle_keys, frequency_ghz, signal_key, error_key
    )
    x_signal, sigma_x_signal = interpolate_angle_sweep(
        x_data, angle_keys, frequency_ghz, signal_key, error_key
    )
    p_co, sigma_p_co = to_detected_power(co_signal, sigma_co_signal, use_power)
    p_x, sigma_p_x = to_detected_power(x_signal, sigma_x_signal, use_power)

    paired = np.isfinite(p_co) & np.isfinite(p_x)
    co_sum = np.sum(p_co[paired]) if np.any(paired) else np.nan
    x_sum = np.sum(p_x[paired]) if np.any(paired) else np.nan

    if use_gain_matching:
        gain_scale = max(abs(co_sum), abs(x_sum), np.finfo(float).tiny)
        gain = (
            co_sum / x_sum
            if np.isfinite(co_sum)
            and np.isfinite(x_sum)
            and abs(x_sum) > np.finfo(float).eps * gain_scale
            else np.nan
        )
    else:
        gain = 1.0

    gain_matched_x = gain * p_x
    denominator = p_co + gain_matched_x
    denominator_scale_values = np.abs(p_co) + np.abs(gain_matched_x)
    finite_scale = denominator_scale_values[np.isfinite(denominator_scale_values)]
    denominator_scale = np.max(finite_scale) if finite_scale.size else np.finfo(float).tiny
    denominator_floor = max(np.finfo(float).tiny, denominator_relative_floor * denominator_scale)

    valid = (
        paired
        & np.isfinite(gain)
        & np.isfinite(denominator)
        & (np.abs(denominator) > denominator_floor)
    )
    d_norm = np.full_like(denominator, np.nan, dtype=float)
    d_norm[valid] = (p_co[valid] - gain_matched_x[valid]) / denominator[valid]

    sigma_d_norm = None
    if sigma_p_co is not None and sigma_p_x is not None:
        sigma_d_norm = np.full_like(d_norm, np.nan, dtype=float)
        uncertainty_valid = (
            valid
            & np.isfinite(sigma_p_co)
            & np.isfinite(sigma_p_x)
            & (sigma_p_co >= 0.0)
            & (sigma_p_x >= 0.0)
        )
        # Derivatives of (a - g b) / (a + g b), treating g as fixed.
        derivative_co = (
            2.0 * gain_matched_x[uncertainty_valid]
            / denominator[uncertainty_valid]**2
        )
        derivative_x = (
            -2.0 * gain * p_co[uncertainty_valid]
            / denominator[uncertainty_valid]**2
        )
        sigma_d_norm[uncertainty_valid] = np.sqrt(
            (derivative_co * sigma_p_co[uncertainty_valid])**2
            + (derivative_x * sigma_p_x[uncertainty_valid])**2
        )

    return {
        'frequency_ghz': float(frequency_ghz),
        'angle_keys': angle_keys,
        'co_signal': co_signal,
        'x_signal': x_signal,
        'p_co': p_co,
        'p_x': p_x,
        'sigma_p_co': sigma_p_co,
        'sigma_p_x': sigma_p_x,
        'gain': float(gain),
        'denominator': denominator,
        'denominator_floor': float(denominator_floor),
        'valid_normalized_mask': valid,
        'n_denominator_floor_failures': int(np.count_nonzero(paired & np.isfinite(denominator) & (np.abs(denominator) <= denominator_floor))),
        'd_norm': d_norm,
        'sigma_d_norm': sigma_d_norm,
    }


def wrap_phase_deg(angle_deg: Iterable[float] | float) -> np.ndarray:
    """Wrap a phase to the interval [-180, 180) deg."""
    return (np.asarray(angle_deg, dtype=float) + 180.0) % 360.0 - 180.0


def wrap_period_deg(angle_deg: Iterable[float] | float, period_deg: float = 90.0) -> np.ndarray:
    """Wrap an angle to [0, period_deg)."""
    return np.mod(np.asarray(angle_deg, dtype=float), period_deg)


def wrap_centered_period_deg(angle_deg: Iterable[float] | float, period_deg: float = 90.0) -> np.ndarray:
    """Wrap a periodic angle difference to [-period/2, period/2)."""
    angle_deg = np.asarray(angle_deg, dtype=float)
    return (angle_deg + 0.5 * period_deg) % period_deg - 0.5 * period_deg


def evaluate_fourier(angles_deg: Iterable[float], c0: float, c4: float, s4: float) -> np.ndarray:
    """Evaluate c0 + c4*cos(4*theta) + s4*sin(4*theta)."""
    theta_rad = np.deg2rad(np.asarray(angles_deg, dtype=float))
    return c0 + c4 * np.cos(4.0 * theta_rad) + s4 * np.sin(4.0 * theta_rad)


def fit_fourier_model(
    angles_deg: Iterable[float],
    values: Iterable[float],
    uncertainties: Iterable[float] | None = None,
) -> dict[str, Any]:
    """Fit constant and fourth-harmonic coefficients and their covariance."""
    angles_deg = np.asarray(angles_deg, dtype=float)
    values = np.asarray(values, dtype=float)
    theta_rad = np.deg2rad(angles_deg)
    design = np.column_stack([
        np.ones_like(theta_rad),
        np.cos(4.0 * theta_rad),
        np.sin(4.0 * theta_rad),
    ])

    base_good = np.isfinite(angles_deg) & np.isfinite(values)
    sigma = None if uncertainties is None else np.asarray(uncertainties, dtype=float)
    sigma_good = (
        np.zeros_like(base_good)
        if sigma is None
        else base_good & np.isfinite(sigma) & (sigma > 0.0)
    )
    
    if (
            sigma is not None
            and np.count_nonzero(base_good) >= MIN_VALID_ANGLES_FOR_FIT
        ):
            use_weighted_fit = True
            good = sigma_good
    else:
            use_weighted_fit = False
            good = base_good
    
    n_valid = int(np.count_nonzero(good))

    empty_result = {
        'c0': np.nan,
        'c4': np.nan,
        's4': np.nan,
        'coefficient_covariance': np.full((3, 3), np.nan),
        'amplitude_4theta': np.nan,
        'amplitude_uncertainty': np.nan,
        'phase_deg': np.nan,
        'model': np.full_like(values, np.nan),
        'residuals': np.full_like(values, np.nan),
        'residual_rms': np.nan,
        'n_valid': n_valid,
        'weighted_fit': use_weighted_fit,
        'fit_success': False,
        'failure_reason': '',
    }
    if n_valid < MIN_VALID_ANGLES_FOR_FIT:
        empty_result['failure_reason'] = f'insufficient finite angle samples: {n_valid}'
        return empty_result
    if np.linalg.matrix_rank(design[good]) < 3:
        empty_result['failure_reason'] = 'rank-deficient fourth-harmonic design matrix'
        return empty_result

    x_fit = design[good]
    y_fit = values[good]
    if use_weighted_fit:
        sqrt_weight = 1.0 / sigma[good]
        x_solve = x_fit * sqrt_weight[:, None]
        y_solve = y_fit * sqrt_weight
        coefficients, *_ = np.linalg.lstsq(x_solve, y_solve, rcond=None)
        covariance = np.linalg.pinv(x_solve.T @ x_solve)
    else:
        coefficients, *_ = np.linalg.lstsq(x_fit, y_fit, rcond=None)
        fit_residuals = y_fit - x_fit @ coefficients
        dof = n_valid - 3
        residual_variance = np.sum(fit_residuals**2) / dof if dof > 0 else np.nan
        covariance = residual_variance * np.linalg.pinv(x_fit.T @ x_fit)

    c0, c4, s4 = coefficients
    model = design @ coefficients
    residuals = values - model
    residual_rms = np.sqrt(np.mean(residuals[good]**2))
    amplitude = np.hypot(c4, s4)
    phase_deg = float(wrap_phase_deg(np.degrees(np.arctan2(s4, c4))))

    amplitude_uncertainty = np.nan
    if amplitude > 0.0 and np.all(np.isfinite(covariance[1:3, 1:3])):
        gradient = np.array([c4, s4], dtype=float) / amplitude
        amplitude_variance = gradient @ covariance[1:3, 1:3] @ gradient
        if np.isfinite(amplitude_variance) and amplitude_variance >= 0.0:
            amplitude_uncertainty = np.sqrt(amplitude_variance)

    return {
        'c0': float(c0),
        'c4': float(c4),
        's4': float(s4),
        'coefficient_covariance': covariance,
        'amplitude_4theta': float(amplitude),
        'amplitude_uncertainty': float(amplitude_uncertainty),
        'phase_deg': phase_deg,
        'model': model,
        'residuals': residuals,
        'residual_rms': float(residual_rms),
        'n_valid': n_valid,
        'weighted_fit': use_weighted_fit,
        'fit_success': True,
        'failure_reason': '',
    }


def hwp_angle_phase_from_raw_fourtheta(
    raw_fourtheta_phase_deg: float,
    sign: float = PHASE_ANGLE_SIGN,
    zero_offset_deg: float = PHASE_ZERO_OFFSET_DEG,
    wrap_period_deg_value: float = PHASE_WRAP_PERIOD_DEG,
) -> float:
    """Convert raw fourth-harmonic phase to wrapped HWP-angle phase."""
    if not np.isfinite(raw_fourtheta_phase_deg):
        return np.nan
    return float(np.mod(sign * raw_fourtheta_phase_deg / 4.0 + zero_offset_deg, wrap_period_deg_value))


def coefficient_uncertainties(covariance: np.ndarray) -> dict[str, float]:
    """Return sqrt diagonal coefficient uncertainties from a 3x3 covariance matrix."""
    covariance = np.asarray(covariance, dtype=float)
    if covariance.shape != (3, 3):
        return {'c0_uncertainty': np.nan, 'c4_uncertainty': np.nan, 's4_uncertainty': np.nan}
    diagonal = np.diag(covariance)
    uncertainties = np.where((diagonal >= 0.0) & np.isfinite(diagonal), np.sqrt(diagonal), np.nan)
    return {
        'c0_uncertainty': float(uncertainties[0]),
        'c4_uncertainty': float(uncertainties[1]),
        's4_uncertainty': float(uncertainties[2]),
    }


def register_figure(filename: str, figure: mpl.figure.Figure) -> mpl.figure.Figure:
    """Register a figure for the automatic export cell."""
    figures_to_save[filename] = figure
    return figure


def add_band_shading(axis: mpl.axes.Axes, include_labels: bool = True) -> None:
    """Shade the PA5/F090 and PA6/F150 designed bands on an axis."""
    axis.axvspan(
        *PA5_F090_BAND_GHZ,
        color=BAND_SHADE_COLOR,
        alpha=0.18,
        label='PA5/F090 (77-112 GHz)' if include_labels else None,
    )
    axis.axvspan(
        *PA6_F150_BAND_GHZ,
        color=BAND_SHADE_COLOR,
        alpha=0.18,
        label='PA6/F150 (124-172 GHz)' if include_labels else None,
    )


print('Transmission/power convention:', TRANSMISSION_POWER_CONVENTION)
print('USE_POWER:', USE_POWER)
print('SIGNAL_ERROR_KEY:', SIGNAL_ERROR_KEY)
print('Phase convention: HWP phase = (PHASE_ANGLE_SIGN * atan2(s4,c4) / 4 + PHASE_ZERO_OFFSET_DEG) mod PHASE_WRAP_PERIOD_DEG')
print(f'PHASE_ANGLE_SIGN={PHASE_ANGLE_SIGN:+.0f}, PHASE_ZERO_OFFSET_DEG={PHASE_ZERO_OFFSET_DEG:g}, PHASE_WRAP_PERIOD_DEG={PHASE_WRAP_PERIOD_DEG:g}')
pipeline_stage_status['helpers_defined'] = True


## 6. Build the normalized angle-frequency data

The 360 deg sample is excluded from all fits when it duplicates the 0 deg physical angle. The full configured angle list is retained separately for inspection and visualization. The default `USE_POWER = False` preserves the current executable notebooks, which treat the selected `SIGNAL_KEY='t'` as a power-like transmission. Change `TRANSMISSION_POWER_CONVENTION` only if a future analyzer output is known to be an electric-field amplitude.


In [ ]:
all_angles_deg = np.asarray(degrees, dtype=float)
all_angle_keys = np.asarray(degrees_str, dtype=object)

wrap_duplicate_angle = START_ANGLE_DEG + PHASE_WRAP_PERIOD_DEG * round((STOP_ANGLE_DEG - START_ANGLE_DEG) / PHASE_WRAP_PERIOD_DEG)
exclude_duplicate_360 = (
    np.isclose((STOP_ANGLE_DEG - START_ANGLE_DEG) % 360.0, 0.0, atol=1e-9)
    and STOP_ANGLE_DEG > START_ANGLE_DEG
)
fit_angle_mask = np.ones_like(all_angles_deg, dtype=bool)
if exclude_duplicate_360:
    fit_angle_mask &= ~np.isclose(all_angles_deg, STOP_ANGLE_DEG, atol=1e-9)

recorded_fit_angles_deg = all_angles_deg[fit_angle_mask]

fit_angles_deg = np.mod(
    ANGLE_DIRECTION_SIGN * recorded_fit_angles_deg,
    360.0,
)

fit_angle_keys = all_angle_keys[fit_angle_mask]

modulation_frequency_grid_ghz = frequency_grid(
    MODULATION_FREQUENCY_START_GHZ,
    MODULATION_FREQUENCY_STOP_GHZ,
    MODULATION_FREQUENCY_STEP_GHZ,
)
phase_fit_freqs_ghz = frequency_grid(
    PHASE_FREQUENCY_START_GHZ,
    PHASE_FREQUENCY_STOP_GHZ,
    PHASE_FREQUENCY_STEP_GHZ,
)

print(f'Configured angle samples: {len(all_angles_deg)}')
print(f'Fitting angle samples: {len(fit_angles_deg)}')
if exclude_duplicate_360:
    print('Excluded the duplicate terminal angle from fitting:', STOP_ANGLE_DEG)
print(f'Modulation frequency samples: {len(modulation_frequency_grid_ghz)}')
print(f'Phase frequency samples: {len(phase_fit_freqs_ghz)}')


def normalized_fit_request_key(frequency_ghz: float, use_gain_matching: bool) -> tuple[Any, ...]:
    """Return a cache key for one normalized-signal fit request."""
    return (
        float(np.round(frequency_ghz, 12)),
        SIGNAL_KEY,
        SIGNAL_ERROR_KEY if SIGNAL_ERROR_KEY is not None else '',
        bool(USE_POWER),
        bool(use_gain_matching),
    )


fit_requests: dict[tuple[Any, ...], dict[str, Any]] = {}


def add_fit_request(frequency_ghz: float, purpose: str, use_gain_matching: bool) -> None:
    """Add or annotate a unique normalized fit request."""
    key = normalized_fit_request_key(frequency_ghz, use_gain_matching)
    if key not in fit_requests:
        fit_requests[key] = {
            'frequency_ghz': float(frequency_ghz),
            'signal_key': SIGNAL_KEY,
            'error_key': SIGNAL_ERROR_KEY,
            'use_power': bool(USE_POWER),
            'use_gain_matching': bool(use_gain_matching),
            'purposes': set(),
        }
    fit_requests[key]['purposes'].add(purpose)


for frequency_ghz in modulation_frequency_grid_ghz:
    add_fit_request(frequency_ghz, 'modulation', MODULATION_USE_GAIN_MATCHING)
for frequency_ghz in phase_fit_freqs_ghz:
    add_fit_request(frequency_ghz, 'phase', PHASE_USE_GAIN_MATCHING)
for frequency_ghz in REPRESENTATIVE_FREQUENCIES_GHZ:
    add_fit_request(float(frequency_ghz), 'modulation_representative', MODULATION_USE_GAIN_MATCHING)
for frequency_ghz in PHASE_REPRESENTATIVE_FREQUENCIES_GHZ:
    add_fit_request(float(frequency_ghz), 'phase_representative', PHASE_USE_GAIN_MATCHING)

normalized_angle_frequency_data: dict[tuple[Any, ...], dict[str, Any]] = {}
normalization_quality_rows: list[dict[str, Any]] = []

for key, request in sorted(fit_requests.items(), key=lambda item: item[1]['frequency_ghz']):
    sweep = construct_normalized_linear_stokes(
        s_data,
        p_data,
        fit_angle_keys,
        request['frequency_ghz'],
        signal_key=request['signal_key'],
        error_key=request['error_key'],
        use_power=request['use_power'],
        use_gain_matching=request['use_gain_matching'],
        denominator_relative_floor=DENOMINATOR_RELATIVE_FLOOR,
    )
    normalized_angle_frequency_data[key] = sweep
    finite_denominator = np.asarray(sweep['denominator'], dtype=float)
    normalization_quality_rows.append({
        'frequency_ghz': request['frequency_ghz'],
        'purposes': ','.join(sorted(request['purposes'])),
        'use_gain_matching': request['use_gain_matching'],
        'gain_factor': sweep['gain'],
        'n_valid_normalized_angles': int(np.count_nonzero(sweep['valid_normalized_mask'])),
        'n_denominator_floor_failures': sweep['n_denominator_floor_failures'],
        'minimum_abs_denominator': float(np.nanmin(np.abs(finite_denominator))) if np.any(np.isfinite(finite_denominator)) else np.nan,
        'denominator_floor': sweep['denominator_floor'],
    })

normalization_quality_summary = pd.DataFrame(normalization_quality_rows)
display(normalization_quality_summary.head())
pipeline_stage_status['normalized_data_built'] = True

print(f'ANGLE_DIRECTION_SIGN: {ANGLE_DIRECTION_SIGN:+.0f}')
print('First recorded fit angles [deg]:', recorded_fit_angles_deg[:6])
print('Corresponding physical fit angles [deg]:', fit_angles_deg[:6])

## 7. Fourth-harmonic fit and central fit table

Every requested normalized-signal fit is represented in `fourth_harmonic_fit_summary`. Modulation-efficiency and Stokes-q phase tables are derived from the same cached coefficients when their signal and gain-matching definitions are the same.


In [ ]:
fourth_harmonic_fit_details: dict[tuple[Any, ...], dict[str, Any]] = {}
fourth_harmonic_rows: list[dict[str, Any]] = []

for key, request in sorted(fit_requests.items(), key=lambda item: item[1]['frequency_ghz']):
    sweep = normalized_angle_frequency_data[key]
    fit = fit_fourier_model(fit_angles_deg, sweep['d_norm'], sweep['sigma_d_norm'])
    fourth_harmonic_fit_details[key] = {
        'request': request,
        'sweep': sweep,
        'fit': fit,
    }
    covariance = np.asarray(fit['coefficient_covariance'], dtype=float)
    coefficient_sigma = coefficient_uncertainties(covariance)
    purposes = set(request['purposes'])
    raw_phase_deg = fit['phase_deg']
    hwp_angle_phase_deg = hwp_angle_phase_from_raw_fourtheta(raw_phase_deg)

    row = {
        'frequency_ghz': request['frequency_ghz'],
        'fit_request_key': repr(key),
        'purposes': ','.join(sorted(purposes)),
        'used_for_modulation': 'modulation' in purposes,
        'used_for_phase': 'phase' in purposes,
        'used_for_modulation_representative_plot': 'modulation_representative' in purposes,
        'used_for_phase_representative_plot': 'phase_representative' in purposes,
        'signal_key': request['signal_key'],
        'error_key': request['error_key'],
        'use_power': request['use_power'],
        'transmission_power_convention': TRANSMISSION_POWER_CONVENTION,
        'use_gain_matching': request['use_gain_matching'],
        'gain_factor': sweep['gain'],
        'n_valid_angle_samples': fit['n_valid'],
        'n_valid_normalized_angles': int(np.count_nonzero(sweep['valid_normalized_mask'])),
        'n_denominator_floor_failures': sweep['n_denominator_floor_failures'],
        'c0': fit['c0'],
        'c4': fit['c4'],
        's4': fit['s4'],
        **coefficient_sigma,
        'cov_c0_c0': float(covariance[0, 0]) if covariance.shape == (3, 3) else np.nan,
        'cov_c0_c4': float(covariance[0, 1]) if covariance.shape == (3, 3) else np.nan,
        'cov_c0_s4': float(covariance[0, 2]) if covariance.shape == (3, 3) else np.nan,
        'cov_c4_c4': float(covariance[1, 1]) if covariance.shape == (3, 3) else np.nan,
        'cov_c4_s4': float(covariance[1, 2]) if covariance.shape == (3, 3) else np.nan,
        'cov_s4_s4': float(covariance[2, 2]) if covariance.shape == (3, 3) else np.nan,
        'fourth_harmonic_amplitude': fit['amplitude_4theta'],
        'fourth_harmonic_amplitude_uncertainty': fit['amplitude_uncertainty'],
        'modulation_efficiency': fit['amplitude_4theta'],
        'modulation_efficiency_uncertainty': fit['amplitude_uncertainty'],
        'raw_fourtheta_phase_deg': raw_phase_deg,
        'hwp_angle_phase_deg': hwp_angle_phase_deg,
        'residual_rms': fit['residual_rms'],
        'weighted_fit': fit['weighted_fit'],
        'fit_success': fit['fit_success'],
        'failure_reason': fit['failure_reason'],
    }
    fourth_harmonic_rows.append(row)

fourth_harmonic_fit_summary = pd.DataFrame(fourth_harmonic_rows).sort_values(
    ['frequency_ghz', 'use_gain_matching', 'purposes']
).reset_index(drop=True)

modulation_efficiency_summary = fourth_harmonic_fit_summary.loc[
    fourth_harmonic_fit_summary['used_for_modulation']
].copy()
modulation_efficiency_summary['n_valid_angles'] = modulation_efficiency_summary['n_valid_angle_samples']
modulation_efficiency_summary['phase_deg'] = modulation_efficiency_summary['raw_fourtheta_phase_deg']
modulation_efficiency_summary = modulation_efficiency_summary[[
    'frequency_ghz', 'n_valid_angles', 'gain_factor',
    'c0', 'c4', 's4', 'c0_uncertainty', 'c4_uncertainty', 's4_uncertainty',
    'modulation_efficiency', 'modulation_efficiency_uncertainty',
    'phase_deg', 'hwp_angle_phase_deg', 'residual_rms', 'weighted_fit',
    'fit_success', 'failure_reason', 'n_denominator_floor_failures',
]]

stokes_q_phase_summary = fourth_harmonic_fit_summary.loc[
    fourth_harmonic_fit_summary['used_for_phase']
].copy()
stokes_q_phase_summary['n_valid_angles'] = stokes_q_phase_summary['n_valid_angle_samples']
stokes_q_phase_summary['q_c0'] = stokes_q_phase_summary['c0']
stokes_q_phase_summary['q_cos4theta_coefficient'] = stokes_q_phase_summary['c4']
stokes_q_phase_summary['q_sin4theta_coefficient'] = stokes_q_phase_summary['s4']
stokes_q_phase_summary['q_modulation_amplitude'] = stokes_q_phase_summary['fourth_harmonic_amplitude']
stokes_q_phase_summary['q_modulation_amplitude_uncertainty'] = stokes_q_phase_summary['fourth_harmonic_amplitude_uncertainty']
stokes_q_phase_summary['q_raw_fourtheta_phase_deg'] = stokes_q_phase_summary['raw_fourtheta_phase_deg']
stokes_q_phase_summary['q_hwp_angle_phase_before_offset_deg'] = (
    PHASE_ANGLE_SIGN * stokes_q_phase_summary['q_raw_fourtheta_phase_deg'] / 4.0
)
stokes_q_phase_summary['q_modulation_peak_angle_deg'] = stokes_q_phase_summary['hwp_angle_phase_deg']
stokes_q_phase_summary['passes_quality_mask'] = (
    stokes_q_phase_summary['fit_success']
    & np.isfinite(stokes_q_phase_summary['q_modulation_peak_angle_deg'])
    & (stokes_q_phase_summary['n_valid_angles'] >= MIN_VALID_ANGLES_FOR_FIT)
    & np.isfinite(stokes_q_phase_summary['q_modulation_amplitude'])
    & (stokes_q_phase_summary['q_modulation_amplitude'] > SMALL_FOURTH_HARMONIC_AMPLITUDE)
    & np.isfinite(stokes_q_phase_summary['residual_rms'])
    & (stokes_q_phase_summary['residual_rms'] <= MAX_RESIDUAL_RMS_WARNING)
)
stokes_q_phase_summary = stokes_q_phase_summary[[
    'frequency_ghz', 'n_valid_angles', 'gain_factor',
    'q_c0', 'q_cos4theta_coefficient', 'q_sin4theta_coefficient',
    'q_modulation_amplitude', 'q_modulation_amplitude_uncertainty',
    'q_raw_fourtheta_phase_deg', 'q_hwp_angle_phase_before_offset_deg',
    'q_modulation_peak_angle_deg', 'residual_rms', 'weighted_fit',
    'fit_success', 'failure_reason', 'passes_quality_mask',
    'n_denominator_floor_failures',
]]

print('Central fourth-harmonic fit rows:', len(fourth_harmonic_fit_summary))
print('Modulation-efficiency rows:', len(modulation_efficiency_summary))
print('Stokes-q phase rows:', len(stokes_q_phase_summary))
display(fourth_harmonic_fit_summary.head())
pipeline_stage_status['fourth_harmonic_fits'] = True


## 8. Modulation-efficiency results

The modulation efficiency is the fitted fourth-harmonic amplitude of the normalized complementary-channel projection. The error bars below are formal fit uncertainties from the coefficient covariance. Band scatter and uncertainty on a band mean are reported separately; they are not interchangeable.


In [ ]:
frequency = modulation_efficiency_summary['frequency_ghz'].to_numpy(dtype=float)
eta_mod = modulation_efficiency_summary['modulation_efficiency'].to_numpy(dtype=float)
sigma_eta = modulation_efficiency_summary['modulation_efficiency_uncertainty'].to_numpy(dtype=float)
valid_error = np.isfinite(sigma_eta) & (sigma_eta > 0.0) & np.isfinite(eta_mod)

fig_main, (ax_eta, ax_residual) = plt.subplots(
    2,
    1,
    figsize=(15, 8),
    sharex=True,
    gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.05},
)

ax_eta.plot(
    frequency,
    eta_mod,
    'o-',
    color=CO_COLOR,
    linewidth=1.8,
    markersize=4,
    label='measured normalized 4theta amplitude',
)
if SHOW_ERROR_BARS and np.any(valid_error):
    ax_eta.errorbar(
        frequency[valid_error],
        eta_mod[valid_error],
        yerr=sigma_eta[valid_error],
        fmt='none',
        ecolor=CO_COLOR,
        elinewidth=0.9,
        capsize=2.0,
        alpha=0.75,
        label='formal fit uncertainty',
    )

ax_eta.axhline(EXPECTED_EFFICIENCY, color='black', linestyle='--', linewidth=1.4, label='ideal expectation')
add_band_shading(ax_eta, include_labels=True)
ax_eta.set_ylabel('Modulation efficiency', fontsize=18)
ax_eta.grid(alpha=0.22)
ax_eta.legend(loc='best')

finite_eta = eta_mod[np.isfinite(eta_mod)]
if finite_eta.size:
    bounds = [finite_eta, np.array([EXPECTED_EFFICIENCY])]
    if np.any(valid_error):
        bounds.extend([eta_mod[valid_error] - sigma_eta[valid_error], eta_mod[valid_error] + sigma_eta[valid_error]])
    all_bounds = np.concatenate(bounds)
    lower, upper = np.nanmin(all_bounds), np.nanmax(all_bounds)
    span = max(upper - lower, 0.10)
    ax_eta.set_ylim(max(0.0, lower - 0.10 * span), upper + 0.10 * span)

eta_residual = eta_mod - EXPECTED_EFFICIENCY
ax_residual.plot(frequency, eta_residual, 'o-', color=CO_COLOR, linewidth=1.4, markersize=3)
if SHOW_ERROR_BARS and np.any(valid_error):
    ax_residual.errorbar(
        frequency[valid_error],
        eta_residual[valid_error],
        yerr=sigma_eta[valid_error],
        fmt='none',
        ecolor=CO_COLOR,
        elinewidth=0.8,
        capsize=2.0,
        alpha=0.65,
    )
ax_residual.axhline(0.0, color='black', linestyle='--', linewidth=1.2)
add_band_shading(ax_residual, include_labels=False)
ax_residual.set_xlabel('Frequency [GHz]', fontsize=18)
ax_residual.set_ylabel(r'$\eta_{\rm mod}-1$', fontsize=18)
ax_residual.grid(alpha=0.22)

register_figure('modulation_efficiency.png', fig_main)
plt.show()


In [ ]:
# Optional measured modulation-efficiency comparison with a digitized Kevin MF AHWP model.
mf_model_summary = pd.DataFrame()
if RUN_MODULATION_MODEL_COMPARISON:
    if Path(MF_MODEL_CSV).exists():
        mf_model_data = pd.read_csv(MF_MODEL_CSV)
        mf_model_frequency_ghz = mf_model_data['frequency_GHz'].to_numpy(dtype=float)
        mf_model_efficiency = mf_model_data['modeled_modulation_efficiency_percent'].to_numpy(dtype=float) / 100.0
        valid_mf_model = np.isfinite(mf_model_frequency_ghz) & np.isfinite(mf_model_efficiency)
        mf_model_frequency_ghz = mf_model_frequency_ghz[valid_mf_model]
        mf_model_efficiency = mf_model_efficiency[valid_mf_model]
        sort_order = np.argsort(mf_model_frequency_ghz)
        mf_model_frequency_ghz = mf_model_frequency_ghz[sort_order]
        mf_model_efficiency = mf_model_efficiency[sort_order]
        mf_model_summary = pd.DataFrame({
            'frequency_ghz': mf_model_frequency_ghz,
            'modeled_modulation_efficiency': mf_model_efficiency,
        })
        print(
            'Kevin MF model frequency range: '
            f'{mf_model_frequency_ghz.min():.2f}-{mf_model_frequency_ghz.max():.2f} GHz'
        )
        print(
            'Kevin MF model efficiency range: '
            f'{mf_model_efficiency.min():.6f}-{mf_model_efficiency.max():.6f}'
        )

        comparison_mask = (
            np.isfinite(frequency)
            & np.isfinite(eta_mod)
            & (frequency >= mf_model_frequency_ghz.min())
            & (frequency <= mf_model_frequency_ghz.max())
        )
        mf_model_at_measured_frequency = np.full_like(eta_mod, np.nan, dtype=float)
        mf_model_at_measured_frequency[comparison_mask] = np.interp(
            frequency[comparison_mask], mf_model_frequency_ghz, mf_model_efficiency
        )
        eta_minus_mf_model = eta_mod - mf_model_at_measured_frequency

        fig_mf, (ax_mf_eta, ax_mf_residual) = plt.subplots(
            2,
            1,
            figsize=(15, 8),
            sharex=True,
            gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.05},
        )
        ax_mf_eta.plot(
            frequency,
            eta_mod,
            'o-',
            color=CO_COLOR,
            linewidth=1.8,
            markersize=4,
            label='measured normalized 4theta amplitude',
        )
        ax_mf_eta.plot(
            mf_model_frequency_ghz,
            mf_model_efficiency,
            '-',
            color=X_COLOR,
            linewidth=2.2,
            label='Kevin MF AHWP model',
        )
        add_band_shading(ax_mf_eta, include_labels=True)
        ax_mf_eta.set_ylabel('Modulation efficiency', fontsize=18)
        ax_mf_eta.set_ylim(0.0, 1.2)
        ax_mf_eta.grid(alpha=0.22)
        ax_mf_eta.legend(loc='best')

        ax_mf_residual.plot(
            frequency[comparison_mask],
            eta_minus_mf_model[comparison_mask],
            'o-',
            color=CO_COLOR,
            linewidth=1.4,
            markersize=3,
            label='measured - Kevin MF model',
        )
        ax_mf_residual.axhline(0.0, color='black', linestyle='--', linewidth=1.2)
        add_band_shading(ax_mf_residual, include_labels=False)
        ax_mf_residual.set_xlabel('Frequency [GHz]', fontsize=18)
        ax_mf_residual.set_ylabel(r'$\eta_{\rm measured}-\eta_{\rm model}$', fontsize=16)
        ax_mf_residual.grid(alpha=0.22)
        ax_mf_residual.legend(loc='best')
        register_figure('modulation_efficiency_kevin_mf_model.png', fig_mf)
        plt.show()
    else:
        warn_pipeline(
            f'Optional Kevin MF modulation model CSV not found: {MF_MODEL_CSV}. '
            'Skipping the Figure 4.21 model overlay.'
        )
else:
    print('Optional modulation model comparison disabled.')


In [ ]:
def find_fit_detail_for_purpose(frequency_ghz: float, purpose: str) -> dict[str, Any]:
    """Return the cached fit detail for an exact frequency and purpose."""
    matches = []
    for key, detail in fourth_harmonic_fit_details.items():
        request = detail['request']
        if purpose in request['purposes'] and np.isclose(request['frequency_ghz'], frequency_ghz, atol=1e-9):
            matches.append(detail)
    if not matches:
        raise KeyError(f'No cached fit detail for {frequency_ghz:g} GHz and purpose {purpose!r}.')
    return matches[0]


dense_angles_deg = np.linspace(0.0, 360.0, 1441)
fig_sweeps, axes = plt.subplots(
    2,
    len(REPRESENTATIVE_FREQUENCIES_GHZ),
    figsize=(15, 7),
    sharex='col',
    gridspec_kw={'height_ratios': [3, 1]},
)

if len(REPRESENTATIVE_FREQUENCIES_GHZ) == 1:
    axes = np.asarray(axes).reshape(2, 1)

for column, requested_frequency in enumerate(REPRESENTATIVE_FREQUENCIES_GHZ):
    details = find_fit_detail_for_purpose(float(requested_frequency), 'modulation_representative')
    sweep = details['sweep']
    fit = details['fit']
    model_dense = evaluate_fourier(dense_angles_deg, fit['c0'], fit['c4'], fit['s4'])

    ax_data = axes[0, column]
    ax_fit_residual = axes[1, column]
    ax_data.plot(
        fit_angles_deg,
        sweep['d_norm'],
        'o',
        color=X_COLOR,
        markersize=3.5,
        alpha=0.80,
        label='measured d_norm',
    )
    ax_data.plot(
        dense_angles_deg,
        model_dense,
        '-',
        color=CO_COLOR,
        linewidth=2.0,
        label='4theta Fourier fit',
    )
    eta_uncertainty_text = (
        f" +/- {fit['amplitude_uncertainty']:.3f}"
        if np.isfinite(fit['amplitude_uncertainty'])
        else ''
    )
    ax_data.set_title(
        f"{requested_frequency:.0f} GHz: eta = {fit['amplitude_4theta']:.3f}{eta_uncertainty_text}, "
        f"raw phase = {fit['phase_deg']:.1f} deg",
        fontsize=13,
    )
    ax_data.set_xlim(0.0, 360.0)
    ax_data.set_xticks([0, 90, 180, 270, 360])
    ax_data.xaxis.set_minor_locator(MultipleLocator(5))
    ax_data.grid(alpha=0.22)

    ax_fit_residual.plot(
        fit_angles_deg,
        fit['residuals'],
        'o',
        color=X_COLOR,
        markersize=3,
        alpha=0.80,
    )
    ax_fit_residual.axhline(0.0, color='black', linestyle='--', linewidth=1.0)
    ax_fit_residual.set_xlim(0.0, 360.0)
    ax_fit_residual.set_xticks([0, 90, 180, 270, 360])
    ax_fit_residual.xaxis.set_minor_locator(MultipleLocator(5))
    ax_fit_residual.set_xlabel('HWP angle [deg]', fontsize=15)
    ax_fit_residual.grid(alpha=0.22)

axes[0, 0].set_ylabel('Normalized linear-Stokes projection', fontsize=15)
axes[1, 0].set_ylabel('Residual', fontsize=15)
axes[0, 0].legend(loc='best')
fig_sweeps.tight_layout()
register_figure('representative_angle_sweeps.png', fig_sweeps)
plt.show()


In [ ]:
fig_diagnostics, diagnostic_axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)

diagnostic_specs = [
    ('gain_factor', 'Relative gain factor', 1.0),
    ('c0', r'Fourier offset $c_0$', 0.0),
    ('residual_rms', 'Residual RMS', None),
    ('phase_deg', 'Raw fitted 4theta phase [deg]', None),
]

for axis, (column, ylabel, reference) in zip(diagnostic_axes.ravel(), diagnostic_specs):
    axis.plot(
        frequency,
        modulation_efficiency_summary[column],
        'o-',
        color=X_COLOR,
        linewidth=1.5,
        markersize=3,
    )
    add_band_shading(axis, include_labels=False)
    if reference is not None:
        axis.axhline(reference, color='black', linestyle='--', linewidth=1.0)
    axis.set_ylabel(ylabel, fontsize=15)
    axis.grid(alpha=0.22)

diagnostic_axes[1, 0].set_xlabel('Frequency [GHz]', fontsize=15)
diagnostic_axes[1, 1].set_xlabel('Frequency [GHz]', fontsize=15)
fig_diagnostics.tight_layout()
register_figure('fit_diagnostics.png', fig_diagnostics)
plt.show()


In [ ]:
def calculate_band_summary(
    modulation_summary: pd.DataFrame,
    designed_band_ghz: tuple[float, float],
    band_name: str,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    """Calculate modulation-efficiency summary statistics within one band."""
    band_low, band_high = designed_band_ghz
    band_summary = modulation_summary.loc[
        modulation_summary['frequency_ghz'].between(band_low, band_high, inclusive='both')
    ].copy()

    band_eta = band_summary['modulation_efficiency'].to_numpy(dtype=float)
    band_sigma = band_summary['modulation_efficiency_uncertainty'].to_numpy(dtype=float)
    band_rms = band_summary['residual_rms'].to_numpy(dtype=float)
    finite_band_eta = np.isfinite(band_eta)
    n_frequencies = int(np.count_nonzero(finite_band_eta))

    unweighted_mean = np.mean(band_eta[finite_band_eta]) if n_frequencies > 0 else np.nan
    frequency_scatter = np.std(band_eta[finite_band_eta], ddof=1) if n_frequencies > 1 else np.nan
    standard_error_unweighted_mean = frequency_scatter / np.sqrt(n_frequencies) if n_frequencies > 1 else np.nan
    minimum_eta = np.min(band_eta[finite_band_eta]) if n_frequencies > 0 else np.nan
    maximum_eta = np.max(band_eta[finite_band_eta]) if n_frequencies > 0 else np.nan

    finite_band_rms = np.isfinite(band_rms)
    mean_residual_rms = np.mean(band_rms[finite_band_rms]) if np.any(finite_band_rms) else np.nan

    valid_inverse_variance = finite_band_eta & np.isfinite(band_sigma) & (band_sigma > 0.0)
    if np.any(valid_inverse_variance):
        inverse_variance_weights = 1.0 / band_sigma[valid_inverse_variance]**2
        inverse_variance_mean = np.average(band_eta[valid_inverse_variance], weights=inverse_variance_weights)
        inverse_variance_mean_uncertainty = np.sqrt(1.0 / np.sum(inverse_variance_weights))
    else:
        inverse_variance_mean = np.nan
        inverse_variance_mean_uncertainty = np.nan

    result = {
        'band_name': band_name,
        'band_low_ghz': band_low,
        'band_high_ghz': band_high,
        'n_frequencies': n_frequencies,
        'unweighted_mean_modulation_efficiency': unweighted_mean,
        'frequency_to_frequency_standard_deviation': frequency_scatter,
        'standard_error_of_unweighted_mean': standard_error_unweighted_mean,
        'inverse_variance_weighted_mean': inverse_variance_mean,
        'formal_uncertainty_of_weighted_mean': inverse_variance_mean_uncertainty,
        'minimum_modulation_efficiency': minimum_eta,
        'maximum_modulation_efficiency': maximum_eta,
        'mean_residual_rms': mean_residual_rms,
    }

    print(f'{band_name}: {band_low:.0f}-{band_high:.0f} GHz')
    print(f'Number of finite frequency samples: {n_frequencies}')
    print(f'Unweighted mean modulation efficiency: {unweighted_mean:.6g}')
    print(f'Frequency-to-frequency standard deviation: {frequency_scatter:.6g}')
    print(f'Standard error of unweighted mean: {standard_error_unweighted_mean:.6g}')
    print(
        'Inverse-variance weighted mean: '
        f'{inverse_variance_mean:.6g} +/- {inverse_variance_mean_uncertainty:.3g} '
        '(formal fit uncertainty)'
    )
    print(f'Minimum / maximum modulation efficiency: {minimum_eta:.6g} / {maximum_eta:.6g}')
    print(f'Mean residual RMS: {mean_residual_rms:.6g}')
    print()

    return band_summary, result


band_summary_f090, band_averaged_result_f090 = calculate_band_summary(
    modulation_efficiency_summary,
    PA5_F090_BAND_GHZ,
    band_name='PA5/F090',
)
band_summary_f150, band_averaged_result_f150 = calculate_band_summary(
    modulation_efficiency_summary,
    PA6_F150_BAND_GHZ,
    band_name='PA6/F150',
)
band_summary = pd.DataFrame([band_averaged_result_f090, band_averaged_result_f150])
display(band_summary)
pipeline_stage_status['modulation_results'] = True


## 9. Normalized Stokes-q phase results

The normalized Stokes-q construction follows Kevin Eq. 4.11, using the relative gain term of Eq. 4.12 when `PHASE_USE_GAIN_MATCHING=True`:

`q(theta, nu) = (P_co(theta, nu) - g(nu) P_x(theta, nu)) / (P_co(theta, nu) + g(nu) P_x(theta, nu))`.

The fitted raw fourth-harmonic phase is `phi_4 = atan2(s4, c4)`. The plotted HWP-angle phase is centralized as:

`theta_q = (PHASE_ANGLE_SIGN * phi_4 / 4 + PHASE_ZERO_OFFSET_DEG) mod PHASE_WRAP_PERIOD_DEG`.

Default `PHASE_ANGLE_SIGN = +1.0` preserves the corrected executed phase notebook and the most complete convention-diagnostic code's numerical result. Setting it to `-1.0` mirrors the stage-coordinate trend; it does not by itself determine the physical viewing side, beam direction, stage rotation direction, or HWP face orientation.


In [ ]:
kevin_phase_frequency = stokes_q_phase_summary['frequency_ghz'].to_numpy(dtype=float)
kevin_hwp_phase = stokes_q_phase_summary['q_modulation_peak_angle_deg'].to_numpy(dtype=float)
phase_quality_mask = stokes_q_phase_summary['passes_quality_mask'].to_numpy(dtype=bool)
valid_kevin_phase = np.isfinite(kevin_phase_frequency) & np.isfinite(kevin_hwp_phase)
low_quality_phase = valid_kevin_phase & ~phase_quality_mask
plot_quality_phase = valid_kevin_phase & phase_quality_mask

fig_kevin_phase, ax_kevin_phase = plt.subplots(1, 1, figsize=(12, 6))

if np.any(plot_quality_phase):
    ax_kevin_phase.plot(
        kevin_phase_frequency[plot_quality_phase],
        kevin_hwp_phase[plot_quality_phase],
        'o-',
        color=CO_COLOR,
        linewidth=1.8,
        markersize=3.5,
        label='Stokes q phase, quality mask pass',
    )
if np.any(low_quality_phase):
    ax_kevin_phase.plot(
        kevin_phase_frequency[low_quality_phase],
        kevin_hwp_phase[low_quality_phase],
        'x',
        color='black',
        markersize=4.0,
        alpha=0.65,
        label='finite fit, quality mask fail',
    )
add_band_shading(ax_kevin_phase, include_labels=True)
ax_kevin_phase.set_xlabel('Frequency [GHz]', fontsize=15)
ax_kevin_phase.set_ylabel('HWP-angle phase [deg]', fontsize=15)
ax_kevin_phase.set_title('Phase for normalized Stokes q parameter', fontsize=15)
ax_kevin_phase.set_xlim(PHASE_FREQUENCY_START_GHZ, PHASE_FREQUENCY_STOP_GHZ)
finite_kevin_hwp_phase = kevin_hwp_phase[valid_kevin_phase]
if finite_kevin_hwp_phase.size:
    lower = np.nanmin(finite_kevin_hwp_phase)
    upper = np.nanmax(finite_kevin_hwp_phase)
    span = max(upper - lower, 5.0)
    ax_kevin_phase.set_ylim(
        max(0.0, lower - 0.12 * span),
        min(PHASE_WRAP_PERIOD_DEG, upper + 0.12 * span),
    )
ax_kevin_phase.grid(alpha=0.22)
ax_kevin_phase.legend(loc='best')

register_figure('stokes_q_phase.png', fig_kevin_phase)
plt.show()


In [ ]:
kevin_dense_angles_deg = np.linspace(0.0, PHASE_WRAP_PERIOD_DEG, 721)
fig_kevin_sweeps, ax_kevin_sweeps = plt.subplots(1, 1, figsize=(10, 6))

for requested_frequency in PHASE_REPRESENTATIVE_FREQUENCIES_GHZ:
    q_details = find_fit_detail_for_purpose(float(requested_frequency), 'phase_representative')
    q_sweep = q_details['sweep']
    q_fit = q_details['fit']
    hwp_angle_phase_deg = hwp_angle_phase_from_raw_fourtheta(q_fit['phase_deg'])

    if not np.isfinite(hwp_angle_phase_deg):
        continue

    q_model_dense = evaluate_fourier(kevin_dense_angles_deg, q_fit['c0'], q_fit['c4'], q_fit['s4'])
    angle_window = (
        np.isfinite(fit_angles_deg)
        & np.isfinite(q_sweep['d_norm'])
        & (fit_angles_deg >= 0.0)
        & (fit_angles_deg <= PHASE_WRAP_PERIOD_DEG)
    )

    ax_kevin_sweeps.plot(fit_angles_deg[angle_window], q_sweep['d_norm'][angle_window], 'o', markersize=3.2, alpha=0.45)
    ax_kevin_sweeps.plot(
        kevin_dense_angles_deg,
        q_model_dense,
        '-',
        linewidth=2.0,
        label=f'{requested_frequency:.0f} GHz, phase = {hwp_angle_phase_deg:.1f} deg',
    )
    ax_kevin_sweeps.axvline(
        hwp_angle_phase_deg,
        color=ax_kevin_sweeps.lines[-1].get_color(),
        linestyle=':',
        linewidth=1.2,
        alpha=0.70,
    )

ax_kevin_sweeps.set_xlim(0.0, PHASE_WRAP_PERIOD_DEG)
ax_kevin_sweeps.set_xticks([0, 15, 30, 45, 60, 75, 90])
ax_kevin_sweeps.xaxis.set_minor_locator(MultipleLocator(5))
ax_kevin_sweeps.set_xlabel('HWP angle [deg]', fontsize=14)
ax_kevin_sweeps.set_ylabel('Stokes q', fontsize=14)
ax_kevin_sweeps.set_title('Normalized Stokes q parameter, 0 to 90 deg', fontsize=15)
ax_kevin_sweeps.grid(alpha=0.22)
ax_kevin_sweeps.legend(loc='best')
fig_kevin_sweeps.tight_layout()
register_figure('representative_stokes_q_curves.png', fig_kevin_sweeps)
plt.show()


In [ ]:
fig_phase_quality, phase_quality_axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
phase_quality_axes[0].plot(
    stokes_q_phase_summary['frequency_ghz'],
    stokes_q_phase_summary['q_modulation_amplitude'],
    'o-',
    color=CO_COLOR,
    linewidth=1.5,
    markersize=3,
)
phase_quality_axes[0].axhline(SMALL_FOURTH_HARMONIC_AMPLITUDE, color='black', linestyle='--', linewidth=1.0)
add_band_shading(phase_quality_axes[0], include_labels=False)
phase_quality_axes[0].set_ylabel('q 4theta amplitude', fontsize=14)
phase_quality_axes[0].grid(alpha=0.22)

phase_quality_axes[1].plot(
    stokes_q_phase_summary['frequency_ghz'],
    stokes_q_phase_summary['residual_rms'],
    'o-',
    color=X_COLOR,
    linewidth=1.5,
    markersize=3,
)
phase_quality_axes[1].axhline(MAX_RESIDUAL_RMS_WARNING, color='black', linestyle='--', linewidth=1.0)
add_band_shading(phase_quality_axes[1], include_labels=False)
phase_quality_axes[1].set_xlabel('Frequency [GHz]', fontsize=14)
phase_quality_axes[1].set_ylabel('Residual RMS', fontsize=14)
phase_quality_axes[1].grid(alpha=0.22)
fig_phase_quality.tight_layout()
register_figure('stokes_q_phase_fit_diagnostics.png', fig_phase_quality)
plt.show()

valid_kevin_phase_rows = stokes_q_phase_summary.loc[valid_kevin_phase].copy()
print('Number of valid Stokes-q frequency samples:', len(valid_kevin_phase_rows))
if len(valid_kevin_phase_rows):
    print(
        'Minimum / maximum valid frequency: '
        f"{valid_kevin_phase_rows['frequency_ghz'].min():.1f} / "
        f"{valid_kevin_phase_rows['frequency_ghz'].max():.1f} GHz"
    )
    print(
        'Raw 4theta phase range: '
        f"{valid_kevin_phase_rows['q_raw_fourtheta_phase_deg'].min():.3g} / "
        f"{valid_kevin_phase_rows['q_raw_fourtheta_phase_deg'].max():.3g} deg"
    )
    print(
        'Configured HWP-angle phase range: '
        f"{valid_kevin_phase_rows['q_modulation_peak_angle_deg'].min():.3g} / "
        f"{valid_kevin_phase_rows['q_modulation_peak_angle_deg'].max():.3g} deg"
    )
else:
    print('Minimum / maximum valid frequency: nan / nan GHz')
    print('Raw 4theta phase range: nan / nan deg')
    print('Configured HWP-angle phase range: nan / nan deg')

print(f'Gain matching used for Stokes q: {PHASE_USE_GAIN_MATCHING}')
print(f'Phase angle sign: {PHASE_ANGLE_SIGN:+.0f}')
print(f'Phase zero offset: {PHASE_ZERO_OFFSET_DEG:.3g} deg')
print(f'Phase wrap period: {PHASE_WRAP_PERIOD_DEG:.3g} deg')
pipeline_stage_status['phase_results'] = True


## 10. Raw-channel and fit-quality diagnostics

These raw-channel Fourier fits are diagnostics only. They are not alternative definitions of physical modulation efficiency. They are useful for checking whether the co-pol and x-pol channels are approximately counterphase, whether residuals are excessive, and whether gain or denominator behavior is unstable.


In [ ]:
# Fit-quality warnings from the shared normalized fits.
failed_core_fits = fourth_harmonic_fit_summary.loc[
    (fourth_harmonic_fit_summary['used_for_modulation'] | fourth_harmonic_fit_summary['used_for_phase'])
    & ~fourth_harmonic_fit_summary['fit_success']
]
if len(failed_core_fits):
    warn_pipeline(f'{len(failed_core_fits)} normalized fourth-harmonic fits failed. See fourth_harmonic_fit_summary.')

insufficient_angle_rows = fourth_harmonic_fit_summary.loc[
    fourth_harmonic_fit_summary['n_valid_angle_samples'] < MIN_VALID_ANGLES_FOR_FIT
]
if len(insufficient_angle_rows):
    warn_pipeline(f'{len(insufficient_angle_rows)} fits have too few valid angle samples.')

excessive_residual_rows = fourth_harmonic_fit_summary.loc[
    fourth_harmonic_fit_summary['fit_success']
    & np.isfinite(fourth_harmonic_fit_summary['residual_rms'])
    & (fourth_harmonic_fit_summary['residual_rms'] > MAX_RESIDUAL_RMS_WARNING)
]
if len(excessive_residual_rows):
    warn_pipeline(
        f'{len(excessive_residual_rows)} normalized fits exceed residual RMS threshold '
        f'{MAX_RESIDUAL_RMS_WARNING:g}.'
    )

small_amplitude_rows = fourth_harmonic_fit_summary.loc[
    fourth_harmonic_fit_summary['fit_success']
    & np.isfinite(fourth_harmonic_fit_summary['fourth_harmonic_amplitude'])
    & (fourth_harmonic_fit_summary['fourth_harmonic_amplitude'] <= SMALL_FOURTH_HARMONIC_AMPLITUDE)
]
if len(small_amplitude_rows):
    warn_pipeline(f'{len(small_amplitude_rows)} normalized fits have very small fourth-harmonic amplitude.')

nonfinite_uncertainty_rows = modulation_efficiency_summary.loc[
    ~np.isfinite(modulation_efficiency_summary['modulation_efficiency_uncertainty'])
]
if len(nonfinite_uncertainty_rows):
    warn_pipeline(f'{len(nonfinite_uncertainty_rows)} modulation frequencies have non-finite formal uncertainties.')

denominator_problem_rows = normalization_quality_summary.loc[
    normalization_quality_summary['n_denominator_floor_failures'] > 0
]
if len(denominator_problem_rows):
    warn_pipeline(
        f'{len(denominator_problem_rows)} normalized sweeps have denominator-floor exclusions. '
        'Inspect normalization_quality_summary.'
    )

finite_gain = fourth_harmonic_fit_summary['gain_factor'].to_numpy(dtype=float)
finite_gain = finite_gain[np.isfinite(finite_gain)]
if finite_gain.size:
    median_gain = float(np.nanmedian(finite_gain))
    if np.isfinite(median_gain) and abs(median_gain) > np.finfo(float).eps:
        relative_gain_deviation = np.abs(fourth_harmonic_fit_summary['gain_factor'] - median_gain) / abs(median_gain)
        unstable_gain_rows = fourth_harmonic_fit_summary.loc[relative_gain_deviation > GAIN_RELATIVE_WARNING]
        if len(unstable_gain_rows):
            warn_pipeline(
                f'{len(unstable_gain_rows)} fit requests have gain factors more than '
                f'{GAIN_RELATIVE_WARNING:.0%} from the median gain.'
            )
else:
    warn_pipeline('All normalized-fit gain factors are non-finite.')

raw_channel_diagnostics = pd.DataFrame()
raw_channel_consistency = pd.DataFrame()

if RUN_RAW_CHANNEL_DIAGNOSTIC:
    raw_rows: list[dict[str, Any]] = []
    raw_channels = {
        'co-pol': {
            'data': s_data,
            'color': CO_COLOR,
            'diagnostic_label': 'co-pol fractional 4theta amplitude',
        },
        'x-pol': {
            'data': p_data,
            'color': X_COLOR,
            'diagnostic_label': 'x-pol fractional 4theta amplitude',
        },
    }

    for frequency_ghz in modulation_frequency_grid_ghz:
        for channel_name, channel_config in raw_channels.items():
            signal, sigma_signal = interpolate_angle_sweep(
                channel_config['data'],
                fit_angle_keys,
                frequency_ghz,
                key=SIGNAL_KEY,
                error_key=SIGNAL_ERROR_KEY,
            )
            power, sigma_power = to_detected_power(signal, sigma_signal, use_power=USE_POWER)
            raw_fit = fit_fourier_model(fit_angles_deg, power, sigma_power)
            fractional_amplitude = (
                raw_fit['amplitude_4theta'] / abs(raw_fit['c0'])
                if np.isfinite(raw_fit['c0']) and raw_fit['c0'] != 0.0
                else np.nan
            )
            raw_rows.append({
                'frequency_ghz': float(frequency_ghz),
                'channel': channel_name,
                'quantity_name': channel_config['diagnostic_label'],
                'c0': raw_fit['c0'],
                'c4': raw_fit['c4'],
                's4': raw_fit['s4'],
                'fourth_harmonic_amplitude': raw_fit['amplitude_4theta'],
                'fractional_4theta_amplitude': fractional_amplitude,
                'phase_deg': raw_fit['phase_deg'],
                'residual_rms': raw_fit['residual_rms'],
                'n_valid_angles': raw_fit['n_valid'],
                'fit_success': raw_fit['fit_success'],
                'failure_reason': raw_fit['failure_reason'],
            })

    raw_channel_diagnostics = pd.DataFrame(raw_rows)
    co_raw = raw_channel_diagnostics.query("channel == 'co-pol'").rename(columns={
        'fractional_4theta_amplitude': 'co_fractional_4theta_amplitude',
        'phase_deg': 'co_phase_deg',
        'residual_rms': 'co_residual_rms',
    })
    x_raw = raw_channel_diagnostics.query("channel == 'x-pol'").rename(columns={
        'fractional_4theta_amplitude': 'x_fractional_4theta_amplitude',
        'phase_deg': 'x_phase_deg',
        'residual_rms': 'x_residual_rms',
    })
    raw_channel_consistency = co_raw[[
        'frequency_ghz', 'co_fractional_4theta_amplitude', 'co_phase_deg', 'co_residual_rms'
    ]].merge(
        x_raw[['frequency_ghz', 'x_fractional_4theta_amplitude', 'x_phase_deg', 'x_residual_rms']],
        on='frequency_ghz',
        how='outer',
    )
    raw_channel_consistency['absolute_phase_separation_deg'] = np.abs(
        wrap_phase_deg(raw_channel_consistency['x_phase_deg'] - raw_channel_consistency['co_phase_deg'])
    )
    raw_channel_consistency['counterphase_residual_from_180_deg'] = np.abs(
        raw_channel_consistency['absolute_phase_separation_deg'] - 180.0
    )
    raw_channel_consistency['passes_counterphase_check'] = (
        np.isfinite(raw_channel_consistency['counterphase_residual_from_180_deg'])
        & (raw_channel_consistency['counterphase_residual_from_180_deg'] <= COUNTERPHASE_TOLERANCE_DEG)
    )

    counterphase_failures = raw_channel_consistency.loc[
        np.isfinite(raw_channel_consistency['counterphase_residual_from_180_deg'])
        & ~raw_channel_consistency['passes_counterphase_check']
    ]
    if len(counterphase_failures):
        warn_pipeline(
            f'{len(counterphase_failures)} raw-channel frequencies fail the counterphase tolerance '
            f'of {COUNTERPHASE_TOLERANCE_DEG:g} deg. See raw_channel_consistency.'
        )

    fig_raw, (ax_fraction, ax_phase_separation) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    for channel_name, channel_config in raw_channels.items():
        channel_rows = raw_channel_diagnostics.query('channel == @channel_name')
        ax_fraction.plot(
            channel_rows['frequency_ghz'],
            channel_rows['fractional_4theta_amplitude'],
            'o-',
            color=channel_config['color'],
            linewidth=1.5,
            markersize=3,
            label=channel_config['diagnostic_label'],
        )

    add_band_shading(ax_fraction, include_labels=False)
    ax_fraction.set_ylabel('Fractional 4theta amplitude', fontsize=15)
    ax_fraction.legend(loc='best')
    ax_fraction.grid(alpha=0.22)

    ax_phase_separation.plot(
        raw_channel_consistency['frequency_ghz'],
        raw_channel_consistency['absolute_phase_separation_deg'],
        'o-',
        color='black',
        linewidth=1.5,
        markersize=3,
    )
    ax_phase_separation.axhline(180.0, color='k', linestyle='--', linewidth=1.2, label='complementary expectation')
    add_band_shading(ax_phase_separation, include_labels=False)
    ax_phase_separation.set_xlabel('Frequency [GHz]', fontsize=15)
    ax_phase_separation.set_ylabel('Absolute phase separation [deg]', fontsize=15)
    ax_phase_separation.grid(alpha=0.22)
    ax_phase_separation.legend(loc='best')
    fig_raw.tight_layout()
    register_figure('raw_channel_diagnostics.png', fig_raw)
    plt.show()
else:
    print('Raw-channel diagnostics disabled by RUN_RAW_CHANNEL_DIAGNOSTIC=False.')

pipeline_stage_status['raw_channel_diagnostics'] = True


## 11. Kevin ideal-model convention diagnostic

This is an idealized Mueller-matrix convention diagnostic based on the Kevin Figure 4.8 analysis. It is separate from the measured modulation-efficiency calculation. The model compares +58 deg and -58 deg central-plate orientations, absolute phase, zero-offset-independent circular trend residuals, and common-reference relative phase.

A mirrored trend match does not independently prove the physical viewing side, beam direction, stage rotation direction, or which HWP face was toward the transmitter. That requires experimental metadata outside this notebook.


In [ ]:
kevin_ideal_phase_model = pd.DataFrame()
kevin_convention_comparison = pd.DataFrame()
kevin_measured_convention_summary = pd.DataFrame()
kevin_peak_position_sanity_checks = pd.DataFrame()

if RUN_KEVIN_MODEL_COMPARISON:
    k48_input_stokes = np.array([1.0, 1.0, 0.0, 0.0], dtype=float)
    k48_nominal_single_plate_frequency_ghz = 120.0
    k48_hwp_angle_grid_deg = np.linspace(0.0, PHASE_WRAP_PERIOD_DEG, 901)
    k48_model_frequency_grid_ghz = np.arange(1.0, 250.0 + 1e-9, 1.0)
    k48_model_plate_angles_plus58_deg = np.array([0.0, 58.0, 0.0], dtype=float)
    k48_model_plate_angles_minus58_deg = np.array([0.0, -58.0, 0.0], dtype=float)

    def k48_mueller_rotation(phi_rad: float) -> np.ndarray:
        """Kevin-style Mueller rotation matrix R(phi)."""
        cos_2phi = np.cos(2.0 * phi_rad)
        sin_2phi = np.sin(2.0 * phi_rad)
        return np.array([
            [1.0, 0.0, 0.0, 0.0],
            [0.0, cos_2phi, -sin_2phi, 0.0],
            [0.0, sin_2phi, cos_2phi, 0.0],
            [0.0, 0.0, 0.0, 1.0],
        ], dtype=float)

    def k48_linear_retarder(delta_rad: float) -> np.ndarray:
        """Linear-retarder Mueller matrix Gamma(delta)."""
        cos_delta = np.cos(delta_rad)
        sin_delta = np.sin(delta_rad)
        return np.array([
            [1.0, 0.0, 0.0, 0.0],
            [0.0, 1.0, 0.0, 0.0],
            [0.0, 0.0, cos_delta, -sin_delta],
            [0.0, 0.0, sin_delta, cos_delta],
        ], dtype=float)

    def k48_rotated_retarder(phi_rad: float, delta_rad: float) -> np.ndarray:
        """Rotated plate using Kevin's R(-phi) @ Gamma(delta) @ R(phi)."""
        return k48_mueller_rotation(-phi_rad) @ k48_linear_retarder(delta_rad) @ k48_mueller_rotation(phi_rad)

    def k48_three_plate_stack_matrix(
        hwp_angle_deg: float,
        relative_plate_angles_deg: np.ndarray,
        delta_rad: float,
    ) -> np.ndarray:
        """Return the Mueller matrix for a rotated three-plate stack."""
        stack_matrix = np.eye(4, dtype=float)
        for relative_angle_deg in relative_plate_angles_deg:
            plate_phi_rad = np.deg2rad(hwp_angle_deg + relative_angle_deg)
            stack_matrix = k48_rotated_retarder(plate_phi_rad, delta_rad) @ stack_matrix
        return stack_matrix

    def k48_three_plate_output_q(
        hwp_angle_deg: float,
        frequency_ghz: float,
        relative_plate_angles_deg: np.ndarray,
    ) -> float:
        """Calculate normalized output q = Q_out / I_out for one angle/frequency."""
        delta_rad = np.pi * float(frequency_ghz) / k48_nominal_single_plate_frequency_ghz
        output_stokes = k48_three_plate_stack_matrix(hwp_angle_deg, relative_plate_angles_deg, delta_rad) @ k48_input_stokes
        if not np.isfinite(output_stokes[0]) or output_stokes[0] == 0.0:
            return np.nan
        return float(output_stokes[1] / output_stokes[0])

    def k48_peak_angle_from_coefficients(c4: float, s4: float) -> float:
        """Return degrees(atan2(s4, c4))/4 wrapped into [0, 90)."""
        if not np.isfinite(c4) or not np.isfinite(s4):
            return np.nan
        return float(np.mod(np.degrees(np.arctan2(s4, c4)) / 4.0, PHASE_WRAP_PERIOD_DEG))

    def k48_fit_q_fourtheta(angle_deg: np.ndarray, q_values: np.ndarray) -> dict[str, Any]:
        """Fit q(theta) = q0 + c4*cos(4theta) + s4*sin(4theta)."""
        angle_deg = np.asarray(angle_deg, dtype=float)
        q_values = np.asarray(q_values, dtype=float)
        good = np.isfinite(angle_deg) & np.isfinite(q_values)
        theta_rad = np.deg2rad(angle_deg)
        design = np.column_stack([
            np.ones_like(theta_rad),
            np.cos(4.0 * theta_rad),
            np.sin(4.0 * theta_rad),
        ])
        empty = {
            'q0': np.nan,
            'c4': np.nan,
            's4': np.nan,
            'amplitude': np.nan,
            'theta_peak_deg': np.nan,
            'n_valid': int(np.count_nonzero(good)),
        }
        if np.count_nonzero(good) < 4 or np.linalg.matrix_rank(design[good]) < 3:
            return empty
        q0, c4, s4 = np.linalg.lstsq(design[good], q_values[good], rcond=None)[0]
        amplitude = np.hypot(c4, s4)
        theta_peak_deg = k48_peak_angle_from_coefficients(c4, s4)
        return {
            'q0': float(q0),
            'c4': float(c4),
            's4': float(s4),
            'amplitude': float(amplitude),
            'theta_peak_deg': theta_peak_deg,
            'n_valid': int(np.count_nonzero(good)),
        }

    def k48_build_three_stack_model(relative_plate_angles_deg: np.ndarray, orientation_label: str) -> pd.DataFrame:
        """Build the idealized Kevin three-stack model summary DataFrame."""
        rows = []
        for frequency_ghz in k48_model_frequency_grid_ghz:
            q_values = np.array([
                k48_three_plate_output_q(hwp_angle_deg, frequency_ghz, relative_plate_angles_deg)
                for hwp_angle_deg in k48_hwp_angle_grid_deg
            ], dtype=float)
            fit = k48_fit_q_fourtheta(k48_hwp_angle_grid_deg, q_values)
            rows.append({
                'frequency_ghz': float(frequency_ghz),
                'model_orientation': orientation_label,
                'central_plate_angle_deg': float(relative_plate_angles_deg[1]),
                'q0': fit['q0'],
                'c4': fit['c4'],
                's4': fit['s4'],
                'amplitude': fit['amplitude'],
                'theta_peak_deg': fit['theta_peak_deg'],
                'n_valid_angles': fit['n_valid'],
            })
        return pd.DataFrame(rows)

    k48_model_plus58_summary = k48_build_three_stack_model(k48_model_plate_angles_plus58_deg, '+58 deg central plate')
    k48_model_minus58_summary = k48_build_three_stack_model(k48_model_plate_angles_minus58_deg, '-58 deg mirrored central plate')
    kevin_ideal_phase_model = pd.concat([k48_model_plus58_summary, k48_model_minus58_summary], ignore_index=True)

    k48_model_mirror_error_deg = wrap_centered_period_deg(
        k48_model_minus58_summary['theta_peak_deg'].to_numpy(dtype=float)
        - np.mod(-k48_model_plus58_summary['theta_peak_deg'].to_numpy(dtype=float), PHASE_WRAP_PERIOD_DEG),
        period_deg=PHASE_WRAP_PERIOD_DEG,
    )
    k48_model_mirror_amplitude_floor = 1e-6
    k48_model_mirror_valid = (
        (k48_model_plus58_summary['amplitude'].to_numpy(dtype=float) > k48_model_mirror_amplitude_floor)
        & (k48_model_minus58_summary['amplitude'].to_numpy(dtype=float) > k48_model_mirror_amplitude_floor)
        & np.isfinite(k48_model_mirror_error_deg)
    )
    k48_max_model_mirror_error_deg = np.nanmax(np.abs(k48_model_mirror_error_deg[k48_model_mirror_valid]))
    assert k48_max_model_mirror_error_deg < 1e-8, (
        'The idealized +58/-58 model phases are not related by the expected mirrored-angle convention; '
        f'max error = {k48_max_model_mirror_error_deg}'
    )

    k48_measured_frequency_ghz = stokes_q_phase_summary['frequency_ghz'].to_numpy(dtype=float)
    k48_measured_c4 = stokes_q_phase_summary['q_cos4theta_coefficient'].to_numpy(dtype=float)
    k48_measured_s4 = stokes_q_phase_summary['q_sin4theta_coefficient'].to_numpy(dtype=float)
    k48_measured_c0 = stokes_q_phase_summary['q_c0'].to_numpy(dtype=float)
    k48_measured_amplitude = stokes_q_phase_summary['q_modulation_amplitude'].to_numpy(dtype=float)
    k48_phase_quality = stokes_q_phase_summary['passes_quality_mask'].to_numpy(dtype=bool)
    k48_measured_stage_peak_deg = np.mod(np.degrees(np.arctan2(k48_measured_s4, k48_measured_c4)) / 4.0, PHASE_WRAP_PERIOD_DEG)
    k48_measured_reversed_peak_deg = np.mod(-k48_measured_stage_peak_deg, PHASE_WRAP_PERIOD_DEG)
    k48_measured_finite_base = (
        np.isfinite(k48_measured_frequency_ghz)
        & np.isfinite(k48_measured_c4)
        & np.isfinite(k48_measured_s4)
        & np.isfinite(k48_measured_stage_peak_deg)
    )
    k48_measured_finite = k48_measured_finite_base & k48_phase_quality if USE_PHASE_QUALITY_MASK_FOR_MODEL_COMPARISON else k48_measured_finite_base

    kevin_measured_convention_summary = pd.DataFrame({
        'frequency_ghz': k48_measured_frequency_ghz,
        'measured_c0': k48_measured_c0,
        'measured_c4': k48_measured_c4,
        'measured_s4': k48_measured_s4,
        'measured_amplitude': k48_measured_amplitude,
        'measured_stage_peak_deg': k48_measured_stage_peak_deg,
        'measured_reversed_peak_deg': k48_measured_reversed_peak_deg,
        'finite_for_convention_diagnostic': k48_measured_finite,
        'phase_quality_mask_pass': k48_phase_quality,
        'quality_mask_applied_to_model_comparison': USE_PHASE_QUALITY_MASK_FOR_MODEL_COMPARISON,
    })

    def k48_evaluate_measured_q_fit(angle_deg: np.ndarray, c0: float, c4: float, s4: float) -> np.ndarray:
        """Evaluate the measured fitted q curve from its Fourier coefficients."""
        angle_rad = np.deg2rad(np.asarray(angle_deg, dtype=float))
        return c0 + c4 * np.cos(4.0 * angle_rad) + s4 * np.sin(4.0 * angle_rad)

    def k48_periodic_abs_difference_deg(angle_a_deg: float, angle_b_deg: float, period_deg: float = PHASE_WRAP_PERIOD_DEG) -> float:
        """Absolute wrapped angular difference for a periodic angle."""
        return float(np.abs(wrap_centered_period_deg(np.asarray(angle_a_deg) - np.asarray(angle_b_deg), period_deg=period_deg)))

    k48_peak_check_targets_ghz = np.array(PHASE_REPRESENTATIVE_FREQUENCIES_GHZ, dtype=float)
    k48_peak_check_angle_grid_deg = np.linspace(0.0, PHASE_WRAP_PERIOD_DEG, 9001)
    k48_peak_check_rows = []
    finite_indices = np.flatnonzero(k48_measured_finite_base)
    if finite_indices.size:
        for target_frequency_ghz in k48_peak_check_targets_ghz:
            nearest_index = finite_indices[np.argmin(np.abs(k48_measured_frequency_ghz[finite_indices] - target_frequency_ghz))]
            frequency_ghz = k48_measured_frequency_ghz[nearest_index]
            model_values = k48_evaluate_measured_q_fit(
                k48_peak_check_angle_grid_deg,
                k48_measured_c0[nearest_index],
                k48_measured_c4[nearest_index],
                k48_measured_s4[nearest_index],
            )
            numerical_peak_angle_deg = k48_peak_check_angle_grid_deg[int(np.nanargmax(model_values))]
            analytic_peak_angle_deg = k48_measured_stage_peak_deg[nearest_index]
            peak_disagreement_deg = k48_periodic_abs_difference_deg(numerical_peak_angle_deg, analytic_peak_angle_deg)
            k48_peak_check_rows.append({
                'target_frequency_ghz': target_frequency_ghz,
                'used_frequency_ghz': frequency_ghz,
                'analytic_stage_peak_deg': analytic_peak_angle_deg,
                'numerical_maximum_angle_deg': numerical_peak_angle_deg,
                'periodic_disagreement_deg': peak_disagreement_deg,
            })
        kevin_peak_position_sanity_checks = pd.DataFrame(k48_peak_check_rows)
        max_disagreement = kevin_peak_position_sanity_checks['periodic_disagreement_deg'].max()
        if max_disagreement >= 0.2:
            warn_pipeline(
                f'Measured q peak-position sanity check max disagreement is {max_disagreement:.4g} deg.'
            )
    else:
        warn_pipeline('No finite measured phase samples are available for the Kevin convention diagnostic.')

    fig_k48_absolute, ax_k48_absolute = plt.subplots(1, 1, figsize=(12, 6))
    add_band_shading(ax_k48_absolute, include_labels=True)
    ax_k48_absolute.plot(
        k48_model_plus58_summary['frequency_ghz'],
        k48_model_plus58_summary['theta_peak_deg'],
        '-',
        color='black',
        linewidth=2.0,
        label='ideal Kevin model, +58 deg central plate',
    )
    ax_k48_absolute.plot(
        k48_model_minus58_summary['frequency_ghz'],
        k48_model_minus58_summary['theta_peak_deg'],
        '--',
        color='black',
        linewidth=2.0,
        label='ideal Kevin model, -58 deg mirrored central plate',
    )
    ax_k48_absolute.plot(
        k48_measured_frequency_ghz[k48_measured_finite_base],
        k48_measured_stage_peak_deg[k48_measured_finite_base],
        'o-',
        color=CO_COLOR,
        linewidth=1.5,
        markersize=3.0,
        label='measured q peak, recorded stage coordinate',
    )
    ax_k48_absolute.plot(
        k48_measured_frequency_ghz[k48_measured_finite_base],
        k48_measured_reversed_peak_deg[k48_measured_finite_base],
        'o-',
        color=X_COLOR,
        linewidth=1.5,
        markersize=3.0,
        label='measured q peak, reversed-coordinate alternative',
    )
    ax_k48_absolute.set_xlabel('Frequency [GHz]', fontsize=15)
    ax_k48_absolute.set_ylabel('Phase [deg]', fontsize=15)
    ax_k48_absolute.set_title('Absolute phase-convention comparison, not zero-offset calibrated', fontsize=15)
    ax_k48_absolute.set_xlim(0.0, 250.0)
    ax_k48_absolute.set_ylim(0.0, PHASE_WRAP_PERIOD_DEG)
    ax_k48_absolute.grid(alpha=0.22)
    ax_k48_absolute.legend(loc='best')
    register_figure('kevin_absolute_convention_comparison.png', fig_k48_absolute)
    plt.show()

    def k48_interpolate_periodic_phase_deg(
        source_frequency_ghz: np.ndarray,
        source_phase_deg: np.ndarray,
        target_frequency_ghz: np.ndarray,
        period_deg: float = PHASE_WRAP_PERIOD_DEG,
    ) -> np.ndarray:
        """Interpolate a periodic phase by unwrapping on the full circle."""
        source_frequency_ghz = np.asarray(source_frequency_ghz, dtype=float)
        source_phase_deg = np.asarray(source_phase_deg, dtype=float)
        target_frequency_ghz = np.asarray(target_frequency_ghz, dtype=float)
        finite = np.isfinite(source_frequency_ghz) & np.isfinite(source_phase_deg)
        result = np.full_like(target_frequency_ghz, np.nan, dtype=float)
        if np.count_nonzero(finite) < 2:
            return result
        source_frequency_ghz = source_frequency_ghz[finite]
        source_phase_deg = source_phase_deg[finite]
        order = np.argsort(source_frequency_ghz)
        source_frequency_ghz = source_frequency_ghz[order]
        source_phase_deg = source_phase_deg[order]
        full_circle_rad = np.deg2rad(source_phase_deg * 360.0 / period_deg)
        unwrapped_rad = np.unwrap(full_circle_rad)
        in_range = (
            np.isfinite(target_frequency_ghz)
            & (target_frequency_ghz >= source_frequency_ghz[0])
            & (target_frequency_ghz <= source_frequency_ghz[-1])
        )
        interpolated_rad = np.interp(target_frequency_ghz[in_range], source_frequency_ghz, unwrapped_rad)
        result[in_range] = np.mod(np.rad2deg(interpolated_rad) * period_deg / 360.0, period_deg)
        return result

    def k48_circular_mean_period_deg(angle_deg: np.ndarray, period_deg: float = PHASE_WRAP_PERIOD_DEG) -> float:
        """Circular mean of periodic angles with the specified period."""
        angle_deg = np.asarray(angle_deg, dtype=float)
        finite = np.isfinite(angle_deg)
        if not np.any(finite):
            return np.nan
        full_circle_rad = np.deg2rad(angle_deg[finite] * 360.0 / period_deg)
        mean_vector = np.mean(np.exp(1j * full_circle_rad))
        if np.abs(mean_vector) <= np.finfo(float).eps:
            return np.nan
        return float(wrap_centered_period_deg(np.rad2deg(np.angle(mean_vector)) * period_deg / 360.0, period_deg=period_deg))

    def k48_compare_model_to_measured_trend(
        model_summary: pd.DataFrame,
        orientation_label: str,
        range_name: str,
        range_mask: np.ndarray,
    ) -> dict[str, Any]:
        """Fit a constant circular offset and return residual diagnostics."""
        model_phase_at_measured = k48_interpolate_periodic_phase_deg(
            model_summary['frequency_ghz'].to_numpy(dtype=float),
            model_summary['theta_peak_deg'].to_numpy(dtype=float),
            k48_measured_frequency_ghz,
            period_deg=PHASE_WRAP_PERIOD_DEG,
        )
        valid = range_mask & k48_measured_finite & np.isfinite(model_phase_at_measured)
        n_points = int(np.count_nonzero(valid))
        if n_points == 0:
            return {
                'comparison_range': range_name,
                'model_orientation': orientation_label,
                'fitted_constant_offset_deg': np.nan,
                'circular_rms_residual_deg': np.nan,
                'n_finite_frequency_points': 0,
                'quality_mask_applied': USE_PHASE_QUALITY_MASK_FOR_MODEL_COMPARISON,
            }
        fitted_offset_deg = k48_circular_mean_period_deg(
            k48_measured_stage_peak_deg[valid] - model_phase_at_measured[valid],
            period_deg=PHASE_WRAP_PERIOD_DEG,
        )
        residual_deg = wrap_centered_period_deg(
            k48_measured_stage_peak_deg[valid] - (model_phase_at_measured[valid] + fitted_offset_deg),
            period_deg=PHASE_WRAP_PERIOD_DEG,
        )
        circular_rms_deg = float(np.sqrt(np.mean(residual_deg**2)))
        return {
            'comparison_range': range_name,
            'model_orientation': orientation_label,
            'fitted_constant_offset_deg': fitted_offset_deg,
            'circular_rms_residual_deg': circular_rms_deg,
            'n_finite_frequency_points': n_points,
            'quality_mask_applied': USE_PHASE_QUALITY_MASK_FOR_MODEL_COMPARISON,
        }

    k48_full_range_mask = np.isfinite(k48_measured_frequency_ghz)
    k48_pa5_range_mask = k48_full_range_mask & (k48_measured_frequency_ghz >= PA5_F090_BAND_GHZ[0]) & (k48_measured_frequency_ghz <= PA5_F090_BAND_GHZ[1])
    k48_pa6_range_mask = k48_full_range_mask & (k48_measured_frequency_ghz >= PA6_F150_BAND_GHZ[0]) & (k48_measured_frequency_ghz <= PA6_F150_BAND_GHZ[1])
    k48_comparison_ranges = [
        ('full finite measured range', k48_full_range_mask),
        ('PA5 band only', k48_pa5_range_mask),
        ('PA6 band only', k48_pa6_range_mask),
    ]
    k48_model_comparison_inputs = [
        ('+58 deg central plate', k48_model_plus58_summary),
        ('-58 deg mirrored central plate', k48_model_minus58_summary),
    ]

    k48_convention_comparison_rows = []
    for range_name, range_mask in k48_comparison_ranges:
        for orientation_label, model_summary in k48_model_comparison_inputs:
            k48_convention_comparison_rows.append(
                k48_compare_model_to_measured_trend(model_summary, orientation_label, range_name, range_mask)
            )
    kevin_convention_comparison = pd.DataFrame(k48_convention_comparison_rows)
    display(kevin_convention_comparison)

    k48_reference_target_frequency_ghz = 125.0
    k48_valid_reference_indices = np.flatnonzero(k48_measured_finite)
    if k48_valid_reference_indices.size:
        k48_reference_index = k48_valid_reference_indices[
            np.argmin(np.abs(k48_measured_frequency_ghz[k48_valid_reference_indices] - k48_reference_target_frequency_ghz))
        ]
        k48_reference_frequency_ghz = k48_measured_frequency_ghz[k48_reference_index]
        k48_model_plus58_at_measured = k48_interpolate_periodic_phase_deg(
            k48_model_plus58_summary['frequency_ghz'].to_numpy(dtype=float),
            k48_model_plus58_summary['theta_peak_deg'].to_numpy(dtype=float),
            k48_measured_frequency_ghz,
            period_deg=PHASE_WRAP_PERIOD_DEG,
        )
        k48_model_minus58_at_measured = k48_interpolate_periodic_phase_deg(
            k48_model_minus58_summary['frequency_ghz'].to_numpy(dtype=float),
            k48_model_minus58_summary['theta_peak_deg'].to_numpy(dtype=float),
            k48_measured_frequency_ghz,
            period_deg=PHASE_WRAP_PERIOD_DEG,
        )
        k48_relative_measured_stage_deg = wrap_centered_period_deg(
            k48_measured_stage_peak_deg - k48_measured_stage_peak_deg[k48_reference_index],
            period_deg=PHASE_WRAP_PERIOD_DEG,
        )
        k48_relative_model_plus58_deg = wrap_centered_period_deg(
            k48_model_plus58_at_measured - k48_model_plus58_at_measured[k48_reference_index],
            period_deg=PHASE_WRAP_PERIOD_DEG,
        )
        k48_relative_model_minus58_deg = wrap_centered_period_deg(
            k48_model_minus58_at_measured - k48_model_minus58_at_measured[k48_reference_index],
            period_deg=PHASE_WRAP_PERIOD_DEG,
        )

        fig_k48_relative, ax_k48_relative = plt.subplots(1, 1, figsize=(12, 6))
        add_band_shading(ax_k48_relative, include_labels=True)
        ax_k48_relative.axhline(0.0, color='black', linestyle=':', linewidth=1.0)
        ax_k48_relative.plot(
            k48_measured_frequency_ghz[k48_measured_finite],
            k48_relative_measured_stage_deg[k48_measured_finite],
            'o-',
            color=CO_COLOR,
            linewidth=1.5,
            markersize=3.0,
            label='measured stage-coordinate relative phase',
        )
        ax_k48_relative.plot(
            k48_measured_frequency_ghz[k48_measured_finite],
            k48_relative_model_plus58_deg[k48_measured_finite],
            '-',
            color='black',
            linewidth=2.0,
            label='+58 deg model relative phase',
        )
        ax_k48_relative.plot(
            k48_measured_frequency_ghz[k48_measured_finite],
            k48_relative_model_minus58_deg[k48_measured_finite],
            '--',
            color='black',
            linewidth=2.0,
            label='-58 deg mirrored model relative phase',
        )
        ax_k48_relative.set_xlabel('Frequency [GHz]', fontsize=15)
        ax_k48_relative.set_ylabel('Relative phase from reference [deg]', fontsize=15)
        ax_k48_relative.set_title(
            f'Zero-offset-independent phase trends (reference {k48_reference_frequency_ghz:.0f} GHz)',
            fontsize=15,
        )
        ax_k48_relative.set_xlim(0.0, 250.0)
        ax_k48_relative.set_ylim(-0.5 * PHASE_WRAP_PERIOD_DEG, 0.5 * PHASE_WRAP_PERIOD_DEG)
        ax_k48_relative.grid(alpha=0.22)
        ax_k48_relative.legend(loc='best')
        register_figure('kevin_relative_trend_comparison.png', fig_k48_relative)
        plt.show()
    else:
        warn_pipeline('No finite measured phase samples are available for the Kevin relative trend plot.')

    if not kevin_convention_comparison.empty:
        full_summary = kevin_convention_comparison.query("comparison_range == 'full finite measured range'").set_index('model_orientation')
        if {'+58 deg central plate', '-58 deg mirrored central plate'}.issubset(full_summary.index):
            plus58_rms = full_summary.loc['+58 deg central plate', 'circular_rms_residual_deg']
            minus58_rms = full_summary.loc['-58 deg mirrored central plate', 'circular_rms_residual_deg']
            rms_difference_deg = abs(plus58_rms - minus58_rms)
            similarity_threshold_deg = max(0.5, 0.10 * np.nanmin([plus58_rms, minus58_rms]))
            print('Convention diagnostic interpretation:')
            if np.isfinite(plus58_rms) and np.isfinite(minus58_rms) and rms_difference_deg <= similarity_threshold_deg:
                print('The simplified model does not clearly distinguish the +58 and -58 orientation conventions for the measured stage-coordinate trend.')
            elif np.isfinite(plus58_rms) and (not np.isfinite(minus58_rms) or plus58_rms < minus58_rms):
                print('The measured stage-coordinate trend is more consistent with the +58 central-plate orientation in this idealized model.')
            elif np.isfinite(minus58_rms):
                print('The measured stage-coordinate trend is more consistent with the mirrored -58 central-plate orientation in this idealized model.')
            else:
                print('The comparison did not produce finite residuals, so no convention preference can be inferred.')
    print(
        'This does not independently prove physical viewing side, beam direction, stage rotation direction, '
        'which HWP face faced the transmitter, or plate/notch orientation.'
    )
else:
    print('Kevin Figure 4.8 model comparison disabled by RUN_KEVIN_MODEL_COMPARISON=False.')

pipeline_stage_status['kevin_model_comparison'] = True


## 12. Automatic exports

This cell writes run-specific CSV tables, publication-quality PNG figures, and a JSON manifest to `results/<RUN_NAME>/`. Figures use 300 dpi and tight bounding boxes when `SAVE_FIGURES=True`.


In [ ]:
def json_default(value: Any) -> Any:
    """JSON serializer for pathlib and NumPy/Pandas scalar values."""
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if pd.isna(value):
        return None
    raise TypeError(f'Object of type {type(value).__name__} is not JSON serializable')


def save_dataframe_csv(dataframe: pd.DataFrame, filename: str, save_empty: bool = False) -> Path | None:
    """Save a DataFrame to RESULTS_DIR when it is available."""
    if dataframe is None or (dataframe.empty and not save_empty):
        return None
    path = RESULTS_DIR / filename
    dataframe.to_csv(path, index=False)
    exported_files.append(path)
    return path


def save_registered_figures() -> list[Path]:
    """Save all registered figures if figure saving is enabled."""
    saved: list[Path] = []
    if not SAVE_FIGURES:
        print('Figure saving disabled by SAVE_FIGURES=False.')
        return saved
    for filename, figure in figures_to_save.items():
        path = RESULTS_DIR / filename
        figure.savefig(path, dpi=FIGURE_DPI, bbox_inches='tight')
        exported_files.append(path)
        saved.append(path)
    return saved


def package_version(package_name: str) -> str | None:
    """Return installed package version where practical."""
    try:
        return importlib_metadata.version(package_name)
    except Exception:
        return None


def extract_saved_notebook_preview_tables(source_paths: Iterable[Path]) -> list[dict[str, Any]]:
    """Extract saved HTML DataFrame previews from source notebooks when configured."""
    preview_tables: list[dict[str, Any]] = []
    for source_path in source_paths:
        path = Path(source_path).expanduser()
        if not path.exists():
            warn_pipeline(f'Source notebook path for compatibility comparison not found: {path}')
            continue
        try:
            notebook = json.loads(path.read_text())
        except Exception as exc:
            warn_pipeline(f'Could not read source notebook {path}: {type(exc).__name__}: {exc}')
            continue
        for cell_index, cell in enumerate(notebook.get('cells', [])):
            for output in cell.get('outputs', []):
                html = output.get('data', {}).get('text/html')
                if not html:
                    continue
                try:
                    from io import StringIO
                    tables = pd.read_html(StringIO(''.join(html)))
                except Exception:
                    continue
                for table in tables:
                    preview_tables.append({
                        'source_notebook': path.name,
                        'cell_index': cell_index,
                        'table': table,
                    })
    return preview_tables


def compare_preview_table_to_current(preview: pd.DataFrame, current: pd.DataFrame, label: str) -> dict[str, Any] | None:
    """Compare overlapping numeric columns from a saved notebook preview."""
    if 'frequency_ghz' not in preview.columns or 'frequency_ghz' not in current.columns:
        return None
    source = preview.copy()
    if 'Unnamed: 0' in source.columns:
        source = source.drop(columns=['Unnamed: 0'])
    merged = source.merge(current, on='frequency_ghz', suffixes=('_source_preview', '_current'))
    if merged.empty:
        return None
    numeric_columns = [
        column for column in source.columns
        if column != 'frequency_ghz' and column in current.columns
    ]
    max_abs_diff = {}
    for column in numeric_columns:
        source_values = pd.to_numeric(merged[f'{column}_source_preview'], errors='coerce')
        current_values = pd.to_numeric(merged[f'{column}_current'], errors='coerce')
        diff = np.abs(source_values - current_values)
        if np.any(np.isfinite(diff)):
            max_abs_diff[column] = float(np.nanmax(diff))
    return {
        'comparison_label': label,
        'n_visible_rows_compared': int(len(merged)),
        'compared_columns': ','.join(max_abs_diff.keys()),
        'max_abs_difference_by_column': json.dumps(max_abs_diff, sort_keys=True),
        'note': 'Saved Jupyter display tables may contain only head/tail preview rows, not full source results.',
    }


RESULTS_DIR.mkdir(parents=True, exist_ok=True)

compatibility_preview_comparison = pd.DataFrame()
preview_comparison_rows: list[dict[str, Any]] = []
if SOURCE_NOTEBOOK_PATHS:
    for preview in extract_saved_notebook_preview_tables(SOURCE_NOTEBOOK_PATHS):
        table = preview['table']
        source_name = preview['source_notebook']
        cell_index = preview['cell_index']
        columns = set(table.columns)
        if {'frequency_ghz', 'modulation_efficiency'}.issubset(columns):
            comparison = compare_preview_table_to_current(
                table,
                modulation_efficiency_summary,
                f'{source_name} cell {cell_index} modulation preview',
            )
        elif {'frequency_ghz', 'q_modulation_peak_angle_deg'}.issubset(columns):
            comparison = compare_preview_table_to_current(
                table,
                stokes_q_phase_summary,
                f'{source_name} cell {cell_index} phase preview',
            )
        else:
            comparison = None
        if comparison is not None:
            preview_comparison_rows.append(comparison)
    compatibility_preview_comparison = pd.DataFrame(preview_comparison_rows)
    if not compatibility_preview_comparison.empty:
        display(compatibility_preview_comparison)
        save_dataframe_csv(compatibility_preview_comparison, 'backward_compatibility_preview_comparison.csv')
else:
    print('No SOURCE_NOTEBOOK_PATHS configured; skipping optional saved-output preview comparison.')

save_dataframe_csv(modulation_efficiency_summary, 'modulation_efficiency_summary.csv', save_empty=True)
save_dataframe_csv(stokes_q_phase_summary, 'stokes_q_phase_summary.csv', save_empty=True)
save_dataframe_csv(fourth_harmonic_fit_summary, 'fourth_harmonic_fit_summary.csv', save_empty=True)
save_dataframe_csv(band_summary, 'band_summary.csv', save_empty=True)
save_dataframe_csv(raw_channel_diagnostics, 'raw_channel_diagnostics.csv')
save_dataframe_csv(raw_channel_consistency, 'raw_channel_consistency.csv')
save_dataframe_csv(kevin_ideal_phase_model, 'kevin_ideal_phase_model.csv')
save_dataframe_csv(kevin_convention_comparison, 'kevin_convention_comparison.csv')

saved_figures = save_registered_figures()
pipeline_stage_status['exports'] = True

manifest_path = RESULTS_DIR / 'run_manifest.json'
manifest_export_list = [*exported_files, manifest_path]

run_manifest = {
    'run_name': RUN_NAME,
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'input_paths': {
        'data_root': DATA_ROOT,
        'data_dir': DATA_DIR,
    },
    'output_paths': {
        'results_dir': RESULTS_DIR,
    },
    'filenames_and_naming_rules': {
        'co_pol_filename_stem': CO_POL_FILENAME_STEM,
        'x_pol_filename_stem': X_POL_FILENAME_STEM,
        'open_reference_stem': OPEN_REFERENCE_STEM,
        'open_reference_name': expected_open_reference_base_name(),
        'scan_suffix': SCAN_SUFFIX,
    },
    'angle_sampling': {
        'start_angle_deg': START_ANGLE_DEG,
        'stop_angle_deg': STOP_ANGLE_DEG,
        'angle_step_deg': ANGLE_STEP_DEG,
        'n_configured_angles': int(len(all_angles_deg)),
        'n_fitting_angles': int(len(fit_angles_deg)),
        'excluded_duplicate_terminal_angle': bool(exclude_duplicate_360),
    },
    'frequency_ranges': {
        'modulation_frequency_start_ghz': MODULATION_FREQUENCY_START_GHZ,
        'modulation_frequency_stop_ghz': MODULATION_FREQUENCY_STOP_GHZ,
        'modulation_frequency_step_ghz': MODULATION_FREQUENCY_STEP_GHZ,
        'phase_frequency_start_ghz': PHASE_FREQUENCY_START_GHZ,
        'phase_frequency_stop_ghz': PHASE_FREQUENCY_STOP_GHZ,
        'phase_frequency_step_ghz': PHASE_FREQUENCY_STEP_GHZ,
        'representative_frequencies_ghz': REPRESENTATIVE_FREQUENCIES_GHZ,
        'phase_representative_frequencies_ghz': PHASE_REPRESENTATIVE_FREQUENCIES_GHZ,
    },
    'band_definitions': {
        'PA5_F090_GHz': PA5_F090_BAND_GHZ,
        'PA6_F150_GHz': PA6_F150_BAND_GHZ,
    },
    'signal_convention': {
        'signal_key': SIGNAL_KEY,
        'signal_error_key': SIGNAL_ERROR_KEY,
        'transmission_power_convention': TRANSMISSION_POWER_CONVENTION,
        'use_power': USE_POWER,
        'expected_efficiency': EXPECTED_EFFICIENCY,
    },
    'gain_matching_choices': {
        'modulation_use_gain_matching': MODULATION_USE_GAIN_MATCHING,
        'phase_use_gain_matching': PHASE_USE_GAIN_MATCHING,
    },
    'phase_convention': {
        'phase_angle_sign': PHASE_ANGLE_SIGN,
        'phase_zero_offset_deg': PHASE_ZERO_OFFSET_DEG,
        'phase_wrap_period_deg': PHASE_WRAP_PERIOD_DEG,
        'formula': '(phase_angle_sign * atan2(s4,c4) / 4 + phase_zero_offset_deg) mod phase_wrap_period_deg',
    },
    'analyzer_configuration': {
        'window': TERAFLASH_WINDOW,
        'window_size': TERAFLASH_WINDOW_SIZE,
        'mask_bounds': MASK_BOUNDS,
        'layout': TERAFLASH_LAYOUT,
    },
    'loaded_scan_counts': {
        'co_pol': int(len(s_data)),
        'x_pol': int(len(p_data)),
        'expected_per_channel': int(len(all_angles_deg)),
    },
    'package_versions': {
        'python': sys.version,
        'platform': platform.platform(),
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'matplotlib': mpl.__version__,
        'TeraToptica': package_version('TeraToptica'),
    },
    'warnings': analysis_warnings,
    'pipeline_stage_status': pipeline_stage_status,
    'exported_files': manifest_export_list,
}

manifest_path.write_text(json.dumps(run_manifest, indent=2, default=json_default))
exported_files.append(manifest_path)

print(f'Results directory: {RESULTS_DIR}')
print('Exported files:')
for path in exported_files:
    print(f'  - {path}')

## 13. Final validation and run summary

This final cell reports the required run checks. It only prints a success message when the required data loading, fitting, summary, and export stages have completed.


In [ ]:
expected_scans_per_channel = len(all_angles_deg)
loaded_all_scans = (len(s_data) == expected_scans_per_channel) and (len(p_data) == expected_scans_per_channel)
required_core_exports = [
    RESULTS_DIR / 'modulation_efficiency_summary.csv',
    RESULTS_DIR / 'stokes_q_phase_summary.csv',
    RESULTS_DIR / 'fourth_harmonic_fit_summary.csv',
    RESULTS_DIR / 'band_summary.csv',
    RESULTS_DIR / 'run_manifest.json',
]
core_exports_exist = all(path.exists() for path in required_core_exports)

failed_modulation_fits = modulation_efficiency_summary.loc[~modulation_efficiency_summary['fit_success']]
failed_phase_fits = stokes_q_phase_summary.loc[~stokes_q_phase_summary['fit_success']]

print('Final HWP analysis run summary')
print(f'  Run name: {RUN_NAME}')
print(f'  Data directory: {Path(DATA_DIR).expanduser()}')
print(f'  Results directory: {RESULTS_DIR}')
print(f'  Loaded co-pol scans: {len(s_data)} / {expected_scans_per_channel}')
print(f'  Loaded x-pol scans: {len(p_data)} / {expected_scans_per_channel}')
print(f'  Fitting angles: {len(fit_angles_deg)}')
print(f'  Modulation-frequency samples: {len(modulation_frequency_grid_ghz)}')
print(f'  Phase-frequency samples: {len(phase_fit_freqs_ghz)}')
print(f'  Failed modulation fits: {len(failed_modulation_fits)}')
print(f'  Failed phase fits: {len(failed_phase_fits)}')
print(f'  Warnings recorded: {len(analysis_warnings)}')
if analysis_warnings:
    for warning in analysis_warnings:
        print(f'    - {warning}')

print('  Exported files:')
for path in exported_files:
    print(f'    - {path}')

required_stage_names = [
    'data_discovery',
    'teraflash_loading',
    'helpers_defined',
    'normalized_data_built',
    'fourth_harmonic_fits',
    'modulation_results',
    'phase_results',
    'raw_channel_diagnostics',
    'kevin_model_comparison',
    'exports',
]
required_stages_completed = all(pipeline_stage_status.get(stage, False) for stage in required_stage_names)
analysis_completed_successfully = (
    loaded_all_scans
    and required_stages_completed
    and len(modulation_efficiency_summary) == len(modulation_frequency_grid_ghz)
    and len(stokes_q_phase_summary) == len(phase_fit_freqs_ghz)
    and core_exports_exist
)

if analysis_completed_successfully:
    print('HWP analysis completed successfully.')
else:
    print('HWP analysis did not complete successfully. Review the failed-fit counts, warnings, and stage status above.')
    print('Stage status:', pipeline_stage_status)

print(f'  Angle direction sign: {ANGLE_DIRECTION_SIGN:+.0f}')
